# WARNING
fenotag gpio=1st

# START

In [1]:
import pandas as pd
import numpy as np
import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
import re
import ast
from itertools import product
from scipy.spatial.distance import cdist # Added for cdist

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix, accuracy_score # Added accuracy_score
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import DecisionBoundaryDisplay

from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# from keras.models import Sequential
# from keras.layers import Dense, Dropout
# from keras.utils import to_categorical
# from sklearn.metrics import classification_report, accuracy_score

# from keras.models import Sequential, Model
# from keras.layers import Dense, Dropout, Input, Conv1D, MaxPooling1D, Flatten
# from keras.utils import to_categorical, set_random_seed

# from keras.optimizers import Adam
# from keras.callbacks import ModelCheckpoint, EarlyStopping

#
import tensorflow as tf
from keras.utils import set_random_seed, to_categorical
from keras.models import Sequential, Model
from keras.layers import (Dense, Dropout, Input, Conv1D, Conv2D, MaxPooling1D, MaxPooling2D,
                          Flatten, Reshape, LSTM, SpatialDropout2D, BatchNormalization, concatenate)
from keras.optimizers import Adam
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from keras.regularizers import l2
from keras.constraints import max_norm
#

import matplotlib.pyplot as plt
import seaborn as sns

import pickle
import json
from json import loads, dumps

from scipy.signal import find_peaks

def get_nearest_antenna(points, antennas):
    # Returns index of nearest antenna for each point
    dists = cdist(points, antennas)
    return np.argmin(dists, axis=1)


In [2]:


import sklearn
sklearn.__version__

'1.8.0'

file.split('__')[-1].rstrip('.csv')

In [3]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 25.8 gigabytes of available RAM

You are using a high-RAM runtime!


In [4]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

zsh:1: command not found: nvidia-smi


In [5]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

In [6]:
os.getcwd()

'/Users/melchior/RFID_PROJECT/RTLS_SF_2/notebook'

In [7]:
os.listdir()

['RTLS HQ_old.ipynb', 'RTLS_SF_codex.ipynb', 'RTLS_SF.ipynb']

In [8]:
# os.listdir('drive')

In [9]:
# os.listdir('drive/MyDrive')

In [10]:
# %cd 'drive/MyDrive/RTLS_SF'

In [11]:
os.listdir()

['RTLS HQ_old.ipynb', 'RTLS_SF_codex.ipynb', 'RTLS_SF.ipynb']

# RFID data

pathfiles

# PX2025 & HQ

In [12]:


Pathfiles_dict = {\

      # ESISAR PX small room, square2m
      'ConfID=2025-10-21 14h00__Item=foam__Qtags=10__Carrier=sapiens__Speed=static__Site=esisar__antConf=floor__ antSep=2m__antTilt=0':\
      {'Reader=SF__Session=S0__Power=30dBm__Q=4':\
           [ 'id=2025-10-21 16h00'],\
      },\

      # ESISAR PX small room, colocated antennas in NSEW
      # 'ConfID=2025-11-18__Item=foam__Qtags=10__Carrier=sapiens__Speed=static__Site=esisar__antConf=floor__ antSep=0m__antTilt=45':\
      # {'Reader=SF__Session=S0__Power=30dBm__Q=4':\
      #      [ 'id=2025-11-18 14h00'],\
      #  },\

      # ESISAR PX anechoic chamber, colocated antennas in NSEW
      #  'ConfID=2025-12-02__Item=foam__Qtags=10__Carrier=sapiens__Speed=static__Site=anechoic_esisar__antConf=floor__ antSep=0m__antTilt=45':\
      #  {'Reader=SF__Session=S0__Power=30dBm__Q=6':\
      #       [ 'id=2025-12-02 14h00'],\
      #  },\

      # HQ
      # 'ConfID=2025-12-10__Item=cardboard__Qtags=20__antQ=8__Site=AcceliotIncHQ':\
      # {'Reader=SF__Session=S?__Power=30dBm__Q=?':\
      #      [ 'id=2025-12-10'],\
      # },\
      #'ConfID=2026-01-21__obj=test__Item=cardboard__Qtags=20__antQ=8__Site=AcceliotIncHQ':\
      #{'Reader=SF__Session=S1__Power=31.5dBm':\
      #     [ 'id=2026-01-21 22h'],\
      # },\
                 }


# HQ

In [13]:

Pathfiles_dict = {\

      # HQ
      'ConfID=2026-02-03__site=HQ' :
      {'pgm=AEP_OK':\
          ['id=2026-02-04', 'id=2026-02-05', 'id=2026-02-06'],\
      },\
                  }


# raw data

HQ: runs (x,y,z0), Actuals (z_rel)

In [14]:
RoundStart = pd.DataFrame()
gpio = pd.DataFrame()
Runs = []
tags = pd.DataFrame()
Actuals = pd.DataFrame()

for Conf in Pathfiles_dict.keys():
  #print(Conf)
  for Reader in Pathfiles_dict[Conf].keys():
    #print(Reader)
    settings_Conf_Reader = Conf + '__' + Reader
    # print(settings_Conf_Reader)
    for runID in Pathfiles_dict[Conf][Reader]:
      #print(runID)
      settings_Conf_Reader_runID = settings_Conf_Reader + '__' + runID

      pathfile = os.path.join ('..','raw_data', Conf, Reader, runID)
      # print(os.listdir(pathfile))
      for file in os.listdir(pathfile):
        print(file)
        if file.endswith('txt'):
      # print(file)
          settings = settings_Conf_Reader_runID + '__' + file.rstrip('.txt')
          #print(settings)
          settings = [x.split('=') for x in settings.split('__') ]
          #print(settings)
          settings = {key:value for key, value in settings}
      # print(settings)
          filename = os.path.join(pathfile, file)
          errors_json=0
          counter_lines=0
          with open(filename, 'rt') as lines:
            run=0
            tags_run=[]
            RoundStart_run=[]
            gpio_run=[]

            for line in lines:
              if line.startswith('data: '):
                data_str = line.split('data: ')[1]
                try:
                  data_json = json.loads(data_str)
                  if run==0:
                    run=data_json['datestamp']
                    run=pd.to_datetime(run, format='%y/%m/%d %H:%M:%S.%f').round('1s')
                    #print(run)
                    settings['run']=run

                  if data_json['type']=='RoundStart':
                    RoundStart_run.append(data_json)
                  if data_json['type']=='TagReadData':
                    tags_run.append(data_json)
                  if data_json['type']=='SensorData':
                    gpio_run.append(data_json)
                except:
                  errors_json += 1
        # print(tags_run)
            tags_run = pd.DataFrame(tags_run)
            tags_run['run'] = run
        # ConfID = settings['ConfID']
        # tags_run['ConfID'] = ConfID
    #
        # for key, value in settings.items():
        #   tags_run[key] = value

            Runs.append(settings)

            tags = pd.concat([tags, tags_run])
            RoundStart_run = pd.DataFrame(RoundStart_run)
            RoundStart_run['run'] = run
        # RoundStart_run['ConfID'] = ConfID
            RoundStart = pd.concat([RoundStart, RoundStart_run])
            gpio_run = pd.DataFrame(gpio_run)
            gpio_run['run'] = run
        # gpio_run['ConfID'] = ConfID
            gpio = pd.concat([gpio, gpio_run])

      #print(errors_json)
  #
  # Actuals
  #
  pathfile_Actuals = os.path.join ('..','raw_data', Conf, 'Actuals')
  #print(pathfile_Actuals)

  for file in os.listdir(pathfile_Actuals):
    print(file)
    filename = os.path.join(pathfile_Actuals, file)
    settings_temp = {x.split('=')[0] : x.split('=')[1]  for x in file.rstrip('.csv').split('__')}
    # Actuals with labelled columns (EPC, x, y, z) or without label (EPC list only )
    Actuals_temp = pd.read_csv(filename, sep=';', dtype=str, decimal=',')
    for key, value in settings_temp.items():
      Actuals_temp[key] = value
    ConfID=settings['ConfID']
    Actuals_temp['ConfID']=ConfID
    Actuals = pd.concat([Actuals, Actuals_temp])
  print(Actuals.shape)
  #
  # antConf
  #
  pathfile_antConf = os.path.join ('..','raw_data', Conf, 'antConf')
  file = os.listdir(pathfile_antConf)[0]
  filename = os.path.join(pathfile_antConf, file)
  #print(filename)
  antConf = pd.read_csv(filename, sep=';', decimal=',')
  # print(antConf.shape)

x=12.91__y=3.39__z0=0__P=24.txt
x=12.91__y=3.39__z0=0__P=27.txt
x=14.53__y=1.52__z0=0__P=31.5.txt
x=8.87__y=4.79__z0=0__P=31.5.txt
x=1.82__y=2.51__z0=0__P=31.5.txt
x=7.73__y=4.45__z0=0__P=31.5.txt
x=5.49__y=4.30__z0=0__P=27.txt
x=1.82__y=2.51__z0=0__P=24.txt
x=5.49__y=4.30__z0=0__P=24.txt
x=12.91__y=3.39__z0=0__P=31.5.txt
x=1.82__y=2.51__z0=0__P=27.txt
x=14.53__y=1.52__z0=0__P=27.txt
x=14.53__y=1.52__z0=0__P=24.txt
x=8.50__y=2.55__z0=0__P=27.txt
x=3.76__y=3.33__z0=0__P=24.txt
x=14.30__y=3.59__z0=0__P=24.txt
x=5.49__y=4.30__z0=0__P=31.5.txt
x=3.76__y=3.33__z0=0__P=27.txt
x=8.50__y=2.55__z0=0__P=24.txt
x=11.23__y=3.36__z0=0__P=31.5.txt
x=14.30__y=3.59__z0=0__P=27.txt
x=7.73__y=4.45__z0=0__P=24.txt
x=8.50__y=2.55__z0=0__P=31.5.txt
x=7.73__y=4.45__z0=0__P=27.txt
x=3.76__y=3.33__z0=0__P=31.5.txt
x=11.23__y=3.36__z0=0__P=27.txt
x=11.23__y=3.36__z0=0__P=24.txt
x=8.87__y=4.79__z0=0__P=24.txt
x=14.30__y=3.59__z0=0__P=31.5.txt
x=8.87__y=4.79__z0=0__P=27.txt
x=12.73__y=3.05__z0=0.80__P=31.5.txt
x

# Runs

In [15]:
Runs = pd.DataFrame(Runs)
Runs

,ConfID,site,pgm,id,x,y,z0,P,run
0,2026-02-03,HQ,AEP_OK,2026-02-04,12.91,3.39,0,24,2026-02-04 21:55:44
1,2026-02-03,HQ,AEP_OK,2026-02-04,12.91,3.39,0,27,2026-02-04 21:54:59
2,2026-02-03,HQ,AEP_OK,2026-02-04,14.53,1.52,0,31.5,2026-02-04 21:57:53
3,2026-02-03,HQ,AEP_OK,2026-02-04,8.87,4.79,0,31.5,2026-02-04 21:46:13
4,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0,31.5,2026-02-04 21:23:51
...,...,...,...,...,...,...,...,...,...
85,2026-02-03,HQ,AEP_OK,2026-02-06,5.04,3.83,0.68,27,2026-02-06 22:06:43
86,2026-02-03,HQ,AEP_OK,2026-02-06,4.42,2.39,0.68,27,2026-02-06 22:11:43
87,2026-02-03,HQ,AEP_OK,2026-02-06,3.36,3.56,1.13,24,2026-02-06 22:16:39
88,2026-02-03,HQ,AEP_OK,2026-02-06,8.52,1.93,0.79,24,2026-02-06 21:49:37


In [16]:
Runs['xy'] = Runs[['x', 'y']].apply(lambda x: '__'.join([str(y) for y in x]), axis=1)
Runs['x'] = Runs['x'].astype(float)
Runs['y'] = Runs['y'].astype(float)
Runs['z0'] = Runs['z0'].astype(float)
Runs

,ConfID,site,pgm,id,x,y,z0,P,run,xy
0,2026-02-03,HQ,AEP_OK,2026-02-04,12.91,3.39,0.00,24,2026-02-04 21:55:44,12.91__3.39
1,2026-02-03,HQ,AEP_OK,2026-02-04,12.91,3.39,0.00,27,2026-02-04 21:54:59,12.91__3.39
2,2026-02-03,HQ,AEP_OK,2026-02-04,14.53,1.52,0.00,31.5,2026-02-04 21:57:53,14.53__1.52
3,2026-02-03,HQ,AEP_OK,2026-02-04,8.87,4.79,0.00,31.5,2026-02-04 21:46:13,8.87__4.79
4,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.00,31.5,2026-02-04 21:23:51,1.82__2.51
...,...,...,...,...,...,...,...,...,...,...
85,2026-02-03,HQ,AEP_OK,2026-02-06,5.04,3.83,0.68,27,2026-02-06 22:06:43,5.04__3.83
86,2026-02-03,HQ,AEP_OK,2026-02-06,4.42,2.39,0.68,27,2026-02-06 22:11:43,4.42__2.39
87,2026-02-03,HQ,AEP_OK,2026-02-06,3.36,3.56,1.13,24,2026-02-06 22:16:39,3.36__3.56
88,2026-02-03,HQ,AEP_OK,2026-02-06,8.52,1.93,0.79,24,2026-02-06 21:49:37,8.52__1.93


In [17]:
SN = Runs.groupby(['xy'])['P'].nunique().rename('Ps').reset_index(drop=False)
SN = SN[SN['Ps']!=3]
pd.merge(SN, Runs, on='xy')

,xy,Ps,ConfID,site,pgm,id,x,y,z0,P,run


In [18]:
Runs['run'] = pd.to_datetime(Runs['run'])
Runs = Runs.sort_values('run', ascending=True).reset_index(drop=True)

Runs[:10]

,ConfID,site,pgm,id,x,y,z0,P,run,xy
0,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.0,31.5,2026-02-04 21:23:51,1.82__2.51
1,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.0,27,2026-02-04 21:25:01,1.82__2.51
2,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.0,24,2026-02-04 21:25:58,1.82__2.51
3,2026-02-03,HQ,AEP_OK,2026-02-04,3.76,3.33,0.0,31.5,2026-02-04 21:28:08,3.76__3.33
4,2026-02-03,HQ,AEP_OK,2026-02-04,3.76,3.33,0.0,27,2026-02-04 21:28:55,3.76__3.33
5,2026-02-03,HQ,AEP_OK,2026-02-04,3.76,3.33,0.0,24,2026-02-04 21:29:42,3.76__3.33
6,2026-02-03,HQ,AEP_OK,2026-02-04,5.49,4.30,0.0,31.5,2026-02-04 21:32:13,5.49__4.30
7,2026-02-03,HQ,AEP_OK,2026-02-04,5.49,4.30,0.0,27,2026-02-04 21:33:38,5.49__4.30
8,2026-02-03,HQ,AEP_OK,2026-02-04,5.49,4.30,0.0,24,2026-02-04 21:34:30,5.49__4.30
9,2026-02-03,HQ,AEP_OK,2026-02-04,7.73,4.45,0.0,31.5,2026-02-04 21:38:34,7.73__4.45


In [19]:
tags[:1]

,type,timestamp,datestamp,seqNum,round,txAntennaPort,txExpanderPort,transmitSource,rssi,preambleQuality,decoderQuality,vgaValues,numBits,rxPhasors,digitalGain,data,alias,run
0,TagReadData,1770242143843,26/02/04 21:55:43.843,619157246,58117535,041A9F61,NONE,ENM_3004,"[-63.2, -80.74]",8431,926351,"[221, 152]",128,"[[13509, 28523], [-1069, 15134]]",7.773209,0x3400ACC102000000000000ACC1022413,41A9F6:1,2026-02-04 21:55:44


In [20]:
tags.groupby('run').size().describe()

count      90.000000
mean     2416.500000
std       545.156534
min      1157.000000
25%      2153.500000
50%      2428.500000
75%      2782.750000
max      3591.000000
dtype: float64

In [21]:
Actuals[:1]

,EPC,z_rel,polar,tagID,site,actual,ConfID
0,BA0101000000000000000000,"0,05",H,1,HQ,P,2026-02-03


In [22]:
Actuals['z_rel'] = Actuals['z_rel'].str.replace(',', '.').astype(float)
Actuals.dtypes

EPC           str
z_rel     float64
polar         str
tagID         str
site          str
actual        str
ConfID        str
dtype: object

In [23]:
antConf

,Antenna,x,y,z
0,041A9F60,2.8,2.2,3.6
1,041A9F61,2.7,3.8,3.6
2,PORT_1,5.7,4.0,3.9
3,PORT_2,8.5,3.8,3.9
4,PORT_3,5.6,1.7,3.9
5,PORT_4,8.4,2.1,3.9
6,0F17B312,11.0,1.7,4.1
7,0F17B313,11.2,4.0,4.3


In [24]:
antConf.dtypes

Antenna        str
x          float64
y          float64
z          float64
dtype: object

In [25]:
antConf = antConf.rename(columns={'x': 'x_ant', 'y':'y_ant', 'z':'z_ant'})
antConf

,Antenna,x_ant,y_ant,z_ant
0,041A9F60,2.8,2.2,3.6
1,041A9F61,2.7,3.8,3.6
2,PORT_1,5.7,4.0,3.9
3,PORT_2,8.5,3.8,3.9
4,PORT_3,5.6,1.7,3.9
5,PORT_4,8.4,2.1,3.9
6,0F17B312,11.0,1.7,4.1
7,0F17B313,11.2,4.0,4.3


In [26]:
RoundStart[:1]

,type,timestamp,datestamp,seqNum,inventoryCycleCount,round,freq_MHz,jurisdiction,txAntennaPort,txExpanderPort,transmitSource,txPower_dBm,rxAntennaConfig,validBackchannel,run
0,RoundStart,1770242143875,26/02/04 21:55:43.875,619157248,7,58117536,908.25,US,0F173B12,NONE,ENM_3004,24.0,"{'antennaPort1': 'PORT_3', 'expanderPort1': 'N...",-1,2026-02-04 21:55:44


In [27]:
# weir runs
Runs [~Runs['run'].isin(tags['run'].unique())]

,ConfID,site,pgm,id,x,y,z0,P,run,xy


# tags

## RoundStart rxAntennaConfig

In [28]:
RoundStart['Tx'] = RoundStart[['txAntennaPort', 'txExpanderPort']].apply(lambda x:'__'.join(x), axis=1)
RoundStart['Tx']

0      0F173B12__NONE
1      0F173B13__NONE
2        PORT_1__NONE
3        PORT_2__NONE
4        PORT_3__NONE
            ...      
213    041A9F61__NONE
214    0F173B12__NONE
215    0F173B13__NONE
216      PORT_1__NONE
217      PORT_2__NONE
Name: Tx, Length: 19780, dtype: str

In [29]:
RoundStart.loc[0, 'rxAntennaConfig']

0    {'antennaPort1': 'PORT_3', 'expanderPort1': 'N...
0    {'antennaPort1': 'PORT_1', 'expanderPort1': 'N...
0    {'antennaPort1': 'PORT_2', 'expanderPort1': 'N...
0    {'antennaPort1': 'PORT_2', 'expanderPort1': 'N...
0    {'antennaPort1': 'PORT_3', 'expanderPort1': 'N...
                           ...                        
0    {'antennaPort1': 'PORT_2', 'expanderPort1': 'N...
0    {'antennaPort1': 'PORT_4', 'expanderPort1': 'N...
0    {'antennaPort1': 'PORT_2', 'expanderPort1': 'N...
0    {'antennaPort1': 'PORT_3', 'expanderPort1': 'N...
0    {'antennaPort1': 'PORT_1', 'expanderPort1': 'N...
Name: rxAntennaConfig, Length: 90, dtype: object

In [30]:
RoundStart['rxAntennaConfig'].apply(lambda x:x['antennaPort1']+'__'+x['expanderPort1'])

0      PORT_3__NONE
1      PORT_3__NONE
2      PORT_1__NONE
3      PORT_2__NONE
4      PORT_3__NONE
           ...     
213    PORT_4__NONE
214    PORT_4__NONE
215    PORT_4__NONE
216    PORT_1__NONE
217    PORT_2__NONE
Name: rxAntennaConfig, Length: 19780, dtype: str

In [31]:
RoundStart['rxAntennaConfig'].apply(lambda x:x['antennaPort2']+'__'+x['expanderPort2'])

0      PORT_2__NONE
1      PORT_2__NONE
2      PORT_3__NONE
3      PORT_3__NONE
4      PORT_2__NONE
           ...     
213    PORT_3__NONE
214    PORT_3__NONE
215    PORT_3__NONE
216    PORT_4__NONE
217    PORT_4__NONE
Name: rxAntennaConfig, Length: 19780, dtype: str

In [32]:
RoundStart['rx1_ant'] = RoundStart['rxAntennaConfig'].apply(lambda x:x['antennaPort1']+'__'+x['expanderPort1'])
RoundStart['rx2_ant'] =RoundStart['rxAntennaConfig'].apply(lambda x:x['antennaPort2']+'__'+x['expanderPort2'])
RoundStart['rx1_rx2_ant'] = RoundStart['rx1_ant'] +'__' +  RoundStart['rx2_ant']
RoundStart[:1]

,type,timestamp,datestamp,seqNum,inventoryCycleCount,round,freq_MHz,jurisdiction,txAntennaPort,txExpanderPort,transmitSource,txPower_dBm,rxAntennaConfig,validBackchannel,run,Tx,rx1_ant,rx2_ant,rx1_rx2_ant
0,RoundStart,1770242143875,26/02/04 21:55:43.875,619157248,7,58117536,908.25,US,0F173B12,NONE,ENM_3004,24.0,"{'antennaPort1': 'PORT_3', 'expanderPort1': 'N...",-1,2026-02-04 21:55:44,0F173B12__NONE,PORT_3__NONE,PORT_2__NONE,PORT_3__NONE__PORT_2__NONE


In [33]:
tags[:1]

,type,timestamp,datestamp,seqNum,round,txAntennaPort,txExpanderPort,transmitSource,rssi,preambleQuality,decoderQuality,vgaValues,numBits,rxPhasors,digitalGain,data,alias,run
0,TagReadData,1770242143843,26/02/04 21:55:43.843,619157246,58117535,041A9F61,NONE,ENM_3004,"[-63.2, -80.74]",8431,926351,"[221, 152]",128,"[[13509, 28523], [-1069, 15134]]",7.773209,0x3400ACC102000000000000ACC1022413,41A9F6:1,2026-02-04 21:55:44


In [34]:
tags.columns

Index(['type', 'timestamp', 'datestamp', 'seqNum', 'round', 'txAntennaPort',
       'txExpanderPort', 'transmitSource', 'rssi', 'preambleQuality',
       'decoderQuality', 'vgaValues', 'numBits', 'rxPhasors', 'digitalGain',
       'data', 'alias', 'run'],
      dtype='str')

In [35]:
def func(run, df):
  RoundStart_run = RoundStart [ RoundStart['run'] == run]
  df = df.copy()
  df['run'] = run
  df = pd.merge(df, RoundStart_run[['round', 'Tx','rx1_ant', 'rx2_ant', 'rx1_rx2_ant', 'txPower_dBm', 'freq_MHz']], on='round')
  return df

tags = pd.concat([func(run, df) for run, df in tags.groupby('run')], ignore_index=True)
tags.head()


,type,timestamp,datestamp,seqNum,round,txAntennaPort,txExpanderPort,transmitSource,rssi,preambleQuality,...,digitalGain,data,alias,run,Tx,rx1_ant,rx2_ant,rx1_rx2_ant,txPower_dBm,freq_MHz
0,TagReadData,1770240230645,26/02/04 21:23:50.645,619014168,58106561,041A9F60,NONE,ENM_3004,"[-63.88, -62.82]",7799,...,8.403122,0x3400ACC102000000000000ACC1022413,41A9F6:0,2026-02-04 21:23:51,041A9F60__NONE,PORT_3__NONE,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75
1,TagReadData,1770240230646,26/02/04 21:23:50.646,619014169,58106561,041A9F60,NONE,ENM_3004,"[-74.51, -78.19]",6324,...,10.363052,0x3112323032302D322D32393034377F6A,41A9F6:0,2026-02-04 21:23:51,041A9F60__NONE,PORT_3__NONE,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75
2,TagReadData,1770240230647,26/02/04 21:23:50.647,619014170,58106561,041A9F60,NONE,ENM_3004,"[-77.97, -83.32]",3504,...,18.703186,0x3400BA05070000000000000000008BEF,41A9F6:0,2026-02-04 21:23:51,041A9F60__NONE,PORT_3__NONE,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75
3,TagReadData,1770240230647,26/02/04 21:23:50.647,619014171,58106561,041A9F60,NONE,ENM_3004,"[-76.42, -75.39]",7122,...,9.201904,0x3400BA060400000000000000000042DA,41A9F6:0,2026-02-04 21:23:51,041A9F60__NONE,PORT_3__NONE,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75
4,TagReadData,1770240230648,26/02/04 21:23:50.648,619014172,58106561,041A9F60,NONE,ENM_3004,"[-76.44, -71.35]",7557,...,8.672211,0x3400BA0508000000000000000000DD5E,41A9F6:0,2026-02-04 21:23:51,041A9F60__NONE,PORT_3__NONE,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75


In [36]:
tags.shape

(216956, 24)

## 3 values per detection: rssi1, rssi2, Delta_rssi

In [37]:
tags['rssi1 (dBm)'] = tags['rssi'].apply(lambda x:x[0])
tags['rssi1'] = 10**6 * 10**(tags['rssi1 (dBm)']/10)
# tags['rx1_ant'] = tags['rxAntennaConfig'].apply(lambda x:x[0])

tags['rssi2 (dBm)'] = tags['rssi'].apply(lambda x:x[1])
tags['rssi2'] = 10**6 * 10**(tags['rssi2 (dBm)']/10)
# tags['rx2_ant'] = tags['rxAntennaConfig'].apply(lambda x:x[1])

tags['Delta_rssi (dBm)'] = tags['rssi1 (dBm)'] - tags['rssi2 (dBm)']
tags['Delta_rssi'] = tags['rssi1'] - tags['rssi2']
# tags['rx1_rx2_ant'] = tags['rx1_ant'] +'__' +  tags['rx2_ant']

tags.head()

,type,timestamp,datestamp,seqNum,round,txAntennaPort,txExpanderPort,transmitSource,rssi,preambleQuality,...,rx2_ant,rx1_rx2_ant,txPower_dBm,freq_MHz,rssi1 (dBm),rssi1,rssi2 (dBm),rssi2,Delta_rssi (dBm),Delta_rssi
0,TagReadData,1770240230645,26/02/04 21:23:50.645,619014168,58106561,041A9F60,NONE,ENM_3004,"[-63.88, -62.82]",7799,...,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-63.88,0.409261,-62.82,0.522396,-1.06,-0.113136
1,TagReadData,1770240230646,26/02/04 21:23:50.646,619014169,58106561,041A9F60,NONE,ENM_3004,"[-74.51, -78.19]",6324,...,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-74.51,0.035400,-78.19,0.015171,3.68,0.020229
2,TagReadData,1770240230647,26/02/04 21:23:50.647,619014170,58106561,041A9F60,NONE,ENM_3004,"[-77.97, -83.32]",3504,...,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-77.97,0.015959,-83.32,0.004656,5.35,0.011303
3,TagReadData,1770240230647,26/02/04 21:23:50.647,619014171,58106561,041A9F60,NONE,ENM_3004,"[-76.42, -75.39]",7122,...,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-76.42,0.022803,-75.39,0.028907,-1.03,-0.006103
4,TagReadData,1770240230648,26/02/04 21:23:50.648,619014172,58106561,041A9F60,NONE,ENM_3004,"[-76.44, -71.35]",7557,...,PORT_4__NONE,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-76.44,0.022699,-71.35,0.073282,-5.09,-0.050584


In [38]:
tags[['rssi1 (dBm)', 'rssi2 (dBm)']]

,rssi1 (dBm),rssi2 (dBm)
0,-63.88,-62.82
1,-74.51,-78.19
2,-77.97,-83.32
3,-76.42,-75.39
4,-76.44,-71.35
...,...,...
216951,-63.61,-80.67
216952,-83.54,-70.86
216953,-75.86,-69.35
216954,-78.90,-72.53


In [39]:
tags[['Tx']].drop_duplicates().reset_index(drop=True)

,Tx
0,041A9F60__NONE
1,041A9F61__NONE
2,0F173B12__NONE
3,0F173B13__NONE
4,PORT_1__NONE
5,PORT_2__NONE
6,PORT_3__NONE
7,PORT_4__NONE


In [40]:
tags[['Tx', 'rx1_ant']].drop_duplicates().sort_values(['Tx', 'rx1_ant']).reset_index(drop=True)

,Tx,rx1_ant
0,041A9F60__NONE,PORT_1__NONE
1,041A9F60__NONE,PORT_2__NONE
2,041A9F60__NONE,PORT_3__NONE
3,041A9F60__NONE,PORT_4__NONE
4,041A9F61__NONE,PORT_1__NONE
5,041A9F61__NONE,PORT_2__NONE
6,041A9F61__NONE,PORT_3__NONE
7,041A9F61__NONE,PORT_4__NONE
8,0F173B12__NONE,PORT_1__NONE
9,0F173B12__NONE,PORT_2__NONE


In [41]:
tags[['Tx', 'rx2_ant']].drop_duplicates().sort_values(['Tx', 'rx2_ant']).reset_index(drop=True)

,Tx,rx2_ant
0,041A9F60__NONE,PORT_1__NONE
1,041A9F60__NONE,PORT_2__NONE
2,041A9F60__NONE,PORT_3__NONE
3,041A9F60__NONE,PORT_4__NONE
4,041A9F61__NONE,PORT_1__NONE
5,041A9F61__NONE,PORT_2__NONE
6,041A9F61__NONE,PORT_3__NONE
7,041A9F61__NONE,PORT_4__NONE
8,0F173B12__NONE,PORT_1__NONE
9,0F173B12__NONE,PORT_2__NONE


In [42]:
tags[['Tx', 'rx1_rx2_ant']].drop_duplicates().sort_values(['Tx', 'rx1_rx2_ant']).reset_index(drop=True)

,Tx,rx1_rx2_ant
0,041A9F60__NONE,PORT_1__NONE__PORT_2__NONE
1,041A9F60__NONE,PORT_1__NONE__PORT_3__NONE
2,041A9F60__NONE,PORT_1__NONE__PORT_4__NONE
3,041A9F60__NONE,PORT_2__NONE__PORT_1__NONE
4,041A9F60__NONE,PORT_2__NONE__PORT_3__NONE
5,041A9F60__NONE,PORT_2__NONE__PORT_4__NONE
6,041A9F60__NONE,PORT_3__NONE__PORT_1__NONE
7,041A9F60__NONE,PORT_3__NONE__PORT_2__NONE
8,041A9F60__NONE,PORT_3__NONE__PORT_4__NONE
9,041A9F60__NONE,PORT_4__NONE__PORT_1__NONE


In [43]:
tags['EPC'] = tags['data'].apply(lambda x:x[6:30])
tags['datestamp'] = pd.to_datetime(tags['datestamp'], format='%y/%m/%d %H:%M:%S.%f')
tags

,type,timestamp,datestamp,seqNum,round,txAntennaPort,txExpanderPort,transmitSource,rssi,preambleQuality,...,rx1_rx2_ant,txPower_dBm,freq_MHz,rssi1 (dBm),rssi1,rssi2 (dBm),rssi2,Delta_rssi (dBm),Delta_rssi,EPC
0,TagReadData,1770240230645,2026-02-04 21:23:50.645,619014168,58106561,041A9F60,NONE,ENM_3004,"[-63.88, -62.82]",7799,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-63.88,0.409261,-62.82,0.522396,-1.06,-0.113136,ACC102000000000000ACC102
1,TagReadData,1770240230646,2026-02-04 21:23:50.646,619014169,58106561,041A9F60,NONE,ENM_3004,"[-74.51, -78.19]",6324,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-74.51,0.035400,-78.19,0.015171,3.68,0.020229,323032302D322D3239303437
2,TagReadData,1770240230647,2026-02-04 21:23:50.647,619014170,58106561,041A9F60,NONE,ENM_3004,"[-77.97, -83.32]",3504,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-77.97,0.015959,-83.32,0.004656,5.35,0.011303,BA0507000000000000000000
3,TagReadData,1770240230647,2026-02-04 21:23:50.647,619014171,58106561,041A9F60,NONE,ENM_3004,"[-76.42, -75.39]",7122,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-76.42,0.022803,-75.39,0.028907,-1.03,-0.006103,BA0604000000000000000000
4,TagReadData,1770240230648,2026-02-04 21:23:50.648,619014172,58106561,041A9F60,NONE,ENM_3004,"[-76.44, -71.35]",7557,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-76.44,0.022699,-71.35,0.073282,-5.09,-0.050584,BA0508000000000000000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216951,TagReadData,1770416662394,2026-02-06 22:24:22.394,620092726,58229616,041A9F61,NONE,ENM_3004,"[-63.61, -80.67]",8042,...,PORT_3__NONE__PORT_1__NONE,24.0,926.25,-63.61,0.435512,-80.67,0.008570,17.06,0.426941,ACC102000000000000ACC102
216952,TagReadData,1770416662395,2026-02-06 22:24:22.395,620092727,58229616,041A9F61,NONE,ENM_3004,"[-83.54, -70.86]",6520,...,PORT_3__NONE__PORT_1__NONE,24.0,926.25,-83.54,0.004426,-70.86,0.082035,-12.68,-0.077609,BA0502000000000000000000
216953,TagReadData,1770416662413,2026-02-06 22:24:22.413,620092728,58229616,041A9F61,NONE,ENM_3004,"[-75.86, -69.35]",4214,...,PORT_3__NONE__PORT_1__NONE,24.0,926.25,-75.86,0.025942,-69.35,0.116145,-6.51,-0.090203,BA0510000000000000000000
216954,TagReadData,1770416662433,2026-02-06 22:24:22.433,620092729,58229616,041A9F61,NONE,ENM_3004,"[-78.9, -72.53]",5377,...,PORT_3__NONE__PORT_1__NONE,24.0,926.25,-78.90,0.012882,-72.53,0.055847,-6.37,-0.042965,BA0204000000000000000000


In [44]:
tags

,type,timestamp,datestamp,seqNum,round,txAntennaPort,txExpanderPort,transmitSource,rssi,preambleQuality,...,rx1_rx2_ant,txPower_dBm,freq_MHz,rssi1 (dBm),rssi1,rssi2 (dBm),rssi2,Delta_rssi (dBm),Delta_rssi,EPC
0,TagReadData,1770240230645,2026-02-04 21:23:50.645,619014168,58106561,041A9F60,NONE,ENM_3004,"[-63.88, -62.82]",7799,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-63.88,0.409261,-62.82,0.522396,-1.06,-0.113136,ACC102000000000000ACC102
1,TagReadData,1770240230646,2026-02-04 21:23:50.646,619014169,58106561,041A9F60,NONE,ENM_3004,"[-74.51, -78.19]",6324,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-74.51,0.035400,-78.19,0.015171,3.68,0.020229,323032302D322D3239303437
2,TagReadData,1770240230647,2026-02-04 21:23:50.647,619014170,58106561,041A9F60,NONE,ENM_3004,"[-77.97, -83.32]",3504,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-77.97,0.015959,-83.32,0.004656,5.35,0.011303,BA0507000000000000000000
3,TagReadData,1770240230647,2026-02-04 21:23:50.647,619014171,58106561,041A9F60,NONE,ENM_3004,"[-76.42, -75.39]",7122,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-76.42,0.022803,-75.39,0.028907,-1.03,-0.006103,BA0604000000000000000000
4,TagReadData,1770240230648,2026-02-04 21:23:50.648,619014172,58106561,041A9F60,NONE,ENM_3004,"[-76.44, -71.35]",7557,...,PORT_3__NONE__PORT_4__NONE,31.5,907.75,-76.44,0.022699,-71.35,0.073282,-5.09,-0.050584,BA0508000000000000000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216951,TagReadData,1770416662394,2026-02-06 22:24:22.394,620092726,58229616,041A9F61,NONE,ENM_3004,"[-63.61, -80.67]",8042,...,PORT_3__NONE__PORT_1__NONE,24.0,926.25,-63.61,0.435512,-80.67,0.008570,17.06,0.426941,ACC102000000000000ACC102
216952,TagReadData,1770416662395,2026-02-06 22:24:22.395,620092727,58229616,041A9F61,NONE,ENM_3004,"[-83.54, -70.86]",6520,...,PORT_3__NONE__PORT_1__NONE,24.0,926.25,-83.54,0.004426,-70.86,0.082035,-12.68,-0.077609,BA0502000000000000000000
216953,TagReadData,1770416662413,2026-02-06 22:24:22.413,620092728,58229616,041A9F61,NONE,ENM_3004,"[-75.86, -69.35]",4214,...,PORT_3__NONE__PORT_1__NONE,24.0,926.25,-75.86,0.025942,-69.35,0.116145,-6.51,-0.090203,BA0510000000000000000000
216954,TagReadData,1770416662433,2026-02-06 22:24:22.433,620092729,58229616,041A9F61,NONE,ENM_3004,"[-78.9, -72.53]",5377,...,PORT_3__NONE__PORT_1__NONE,24.0,926.25,-78.90,0.012882,-72.53,0.055847,-6.37,-0.042965,BA0204000000000000000000


In [45]:
tags.columns

Index(['type', 'timestamp', 'datestamp', 'seqNum', 'round', 'txAntennaPort',
       'txExpanderPort', 'transmitSource', 'rssi', 'preambleQuality',
       'decoderQuality', 'vgaValues', 'numBits', 'rxPhasors', 'digitalGain',
       'data', 'alias', 'run', 'Tx', 'rx1_ant', 'rx2_ant', 'rx1_rx2_ant',
       'txPower_dBm', 'freq_MHz', 'rssi1 (dBm)', 'rssi1', 'rssi2 (dBm)',
       'rssi2', 'Delta_rssi (dBm)', 'Delta_rssi', 'EPC'],
      dtype='str')

In [46]:
tags = tags [['run', 'EPC','datestamp', 'Tx', 'txPower_dBm', 'rx1_ant', 'rssi1 (dBm)', 'rssi1', 'rx2_ant', 'rssi2 (dBm)', 'rssi2', 'rx1_rx2_ant', 'Delta_rssi', 'Delta_rssi (dBm)']]
tags

,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),rssi2,rx1_rx2_ant,Delta_rssi,Delta_rssi (dBm)
0,2026-02-04 21:23:51,ACC102000000000000ACC102,2026-02-04 21:23:50.645,041A9F60__NONE,31.5,PORT_3__NONE,-63.88,0.409261,PORT_4__NONE,-62.82,0.522396,PORT_3__NONE__PORT_4__NONE,-0.113136,-1.06
1,2026-02-04 21:23:51,323032302D322D3239303437,2026-02-04 21:23:50.646,041A9F60__NONE,31.5,PORT_3__NONE,-74.51,0.035400,PORT_4__NONE,-78.19,0.015171,PORT_3__NONE__PORT_4__NONE,0.020229,3.68
2,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,0.004656,PORT_3__NONE__PORT_4__NONE,0.011303,5.35
3,2026-02-04 21:23:51,BA0604000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-76.42,0.022803,PORT_4__NONE,-75.39,0.028907,PORT_3__NONE__PORT_4__NONE,-0.006103,-1.03
4,2026-02-04 21:23:51,BA0508000000000000000000,2026-02-04 21:23:50.648,041A9F60__NONE,31.5,PORT_3__NONE,-76.44,0.022699,PORT_4__NONE,-71.35,0.073282,PORT_3__NONE__PORT_4__NONE,-0.050584,-5.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
216951,2026-02-06 22:24:02,ACC102000000000000ACC102,2026-02-06 22:24:22.394,041A9F61__NONE,24.0,PORT_3__NONE,-63.61,0.435512,PORT_1__NONE,-80.67,0.008570,PORT_3__NONE__PORT_1__NONE,0.426941,17.06
216952,2026-02-06 22:24:02,BA0502000000000000000000,2026-02-06 22:24:22.395,041A9F61__NONE,24.0,PORT_3__NONE,-83.54,0.004426,PORT_1__NONE,-70.86,0.082035,PORT_3__NONE__PORT_1__NONE,-0.077609,-12.68
216953,2026-02-06 22:24:02,BA0510000000000000000000,2026-02-06 22:24:22.413,041A9F61__NONE,24.0,PORT_3__NONE,-75.86,0.025942,PORT_1__NONE,-69.35,0.116145,PORT_3__NONE__PORT_1__NONE,-0.090203,-6.51
216954,2026-02-06 22:24:02,BA0204000000000000000000,2026-02-06 22:24:22.433,041A9F61__NONE,24.0,PORT_3__NONE,-78.90,0.012882,PORT_1__NONE,-72.53,0.055847,PORT_3__NONE__PORT_1__NONE,-0.042965,-6.37


## Actuals

In [47]:
Actuals

,EPC,z_rel,polar,tagID,site,actual,ConfID
0,BA0101000000000000000000,0.05,H,1,HQ,P,2026-02-03
1,BA0102000000000000000000,0.05,H,2,HQ,P,2026-02-03
2,BA0103000000000000000000,0.05,H,3,HQ,P,2026-02-03
3,BA0104000000000000000000,0.05,H,4,HQ,P,2026-02-03
4,BA0105000000000000000000,0.05,V,5,HQ,P,2026-02-03
5,BA0106000000000000000000,0.05,V,6,HQ,P,2026-02-03
6,BA0107000000000000000000,0.05,V,7,HQ,P,2026-02-03
7,BA0108000000000000000000,0.05,V,8,HQ,P,2026-02-03
8,BA0109000000000000000000,0.05,H,9,HQ,P,2026-02-03
9,BA0110000000000000000000,0.05,H,10,HQ,P,2026-02-03


In [48]:
Runs[:1]

,ConfID,site,pgm,id,x,y,z0,P,run,xy
0,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.0,31.5,2026-02-04 21:23:51,1.82__2.51


In [49]:
Actuals['key']=1
Runs['key']=1
Actuals_xyz_run = pd.merge(Actuals.drop(columns=['ConfID', 'site']), Runs, on='key')
Actuals_xyz_run

,EPC,z_rel,polar,tagID,actual,key,ConfID,site,pgm,id,x,y,z0,P,run,xy
0,BA0101000000000000000000,0.05,H,1,P,1,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.0,31.5,2026-02-04 21:23:51,1.82__2.51
1,BA0101000000000000000000,0.05,H,1,P,1,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.0,27,2026-02-04 21:25:01,1.82__2.51
2,BA0101000000000000000000,0.05,H,1,P,1,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.0,24,2026-02-04 21:25:58,1.82__2.51
3,BA0101000000000000000000,0.05,H,1,P,1,2026-02-03,HQ,AEP_OK,2026-02-04,3.76,3.33,0.0,31.5,2026-02-04 21:28:08,3.76__3.33
4,BA0101000000000000000000,0.05,H,1,P,1,2026-02-03,HQ,AEP_OK,2026-02-04,3.76,3.33,0.0,27,2026-02-04 21:28:55,3.76__3.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,BA0610000000000000000000,1.67,H,10,P,1,2026-02-03,HQ,AEP_OK,2026-02-06,2.65,3.03,0.0,27,2026-02-06 22:19:48,2.65__3.03
5396,BA0610000000000000000000,1.67,H,10,P,1,2026-02-03,HQ,AEP_OK,2026-02-06,2.65,3.03,0.0,24,2026-02-06 22:20:30,2.65__3.03
5397,BA0610000000000000000000,1.67,H,10,P,1,2026-02-03,HQ,AEP_OK,2026-02-06,3.34,2.01,0.0,31.5,2026-02-06 22:22:32,3.34__2.01
5398,BA0610000000000000000000,1.67,H,10,P,1,2026-02-03,HQ,AEP_OK,2026-02-06,3.34,2.01,0.0,27,2026-02-06 22:23:22,3.34__2.01


In [50]:
Actuals_xyz_run['z'] = Actuals_xyz_run['z0'] + Actuals_xyz_run['z_rel']
Actuals_xyz_run['xyz'] = Actuals_xyz_run[['x', 'y', 'z']].apply(lambda x:'__'.join([str(y) for y in x]), axis=1)
Actuals_xyz_run[['x', 'y', 'z']].describe()

,x,y,z
count,5400.000000,5400.000000,5400.00000
mean,7.948667,3.294667,1.23600
std,4.007642,0.912980,0.71835
min,0.580000,1.260000,0.05000
25%,4.420000,2.550000,0.70000
50%,8.485000,3.375000,1.33000
75%,11.230000,4.080000,1.67000
max,14.530000,4.790000,2.94000


In [51]:
Actuals_xyz_run[:1]

,EPC,z_rel,polar,tagID,actual,key,ConfID,site,pgm,id,x,y,z0,P,run,xy,z,xyz
0,BA0101000000000000000000,0.05,H,1,P,1,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,2.51,0.0,31.5,2026-02-04 21:23:51,1.82__2.51,0.05,1.82__2.51__0.05


In [52]:
Actuals_xyz_run['key']=1
antConf['key']=1
Actuals_xyz_D_run = pd.merge(Actuals_xyz_run, antConf, on='key')
# Actuals_antConf = pd.merge(Actuals, antConf, on='key', suffixes=['', '_ant'])
Actuals_xyz_D_run = Actuals_xyz_D_run.drop(columns=['key'])
Actuals_xyz_D_run

,EPC,z_rel,polar,tagID,actual,ConfID,site,pgm,id,x,...,z0,P,run,xy,z,xyz,Antenna,x_ant,y_ant,z_ant
0,BA0101000000000000000000,0.05,H,1,P,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,...,0.0,31.5,2026-02-04 21:23:51,1.82__2.51,0.05,1.82__2.51__0.05,041A9F60,2.8,2.2,3.6
1,BA0101000000000000000000,0.05,H,1,P,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,...,0.0,31.5,2026-02-04 21:23:51,1.82__2.51,0.05,1.82__2.51__0.05,041A9F61,2.7,3.8,3.6
2,BA0101000000000000000000,0.05,H,1,P,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,...,0.0,31.5,2026-02-04 21:23:51,1.82__2.51,0.05,1.82__2.51__0.05,PORT_1,5.7,4.0,3.9
3,BA0101000000000000000000,0.05,H,1,P,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,...,0.0,31.5,2026-02-04 21:23:51,1.82__2.51,0.05,1.82__2.51__0.05,PORT_2,8.5,3.8,3.9
4,BA0101000000000000000000,0.05,H,1,P,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,...,0.0,31.5,2026-02-04 21:23:51,1.82__2.51,0.05,1.82__2.51__0.05,PORT_3,5.6,1.7,3.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43195,BA0610000000000000000000,1.67,H,10,P,2026-02-03,HQ,AEP_OK,2026-02-06,3.34,...,0.0,24,2026-02-06 22:24:02,3.34__2.01,1.67,3.34__2.01__1.67,PORT_2,8.5,3.8,3.9
43196,BA0610000000000000000000,1.67,H,10,P,2026-02-03,HQ,AEP_OK,2026-02-06,3.34,...,0.0,24,2026-02-06 22:24:02,3.34__2.01,1.67,3.34__2.01__1.67,PORT_3,5.6,1.7,3.9
43197,BA0610000000000000000000,1.67,H,10,P,2026-02-03,HQ,AEP_OK,2026-02-06,3.34,...,0.0,24,2026-02-06 22:24:02,3.34__2.01,1.67,3.34__2.01__1.67,PORT_4,8.4,2.1,3.9
43198,BA0610000000000000000000,1.67,H,10,P,2026-02-03,HQ,AEP_OK,2026-02-06,3.34,...,0.0,24,2026-02-06 22:24:02,3.34__2.01,1.67,3.34__2.01__1.67,0F17B312,11.0,1.7,4.1


In [53]:
Actuals_xyz_D_run['D'] = [np.sqrt((x**2).sum()) for x in (Actuals_xyz_D_run[['x', 'y', 'z']].values - Actuals_xyz_D_run[['x_ant', 'y_ant', 'z_ant']].values)]
Actuals_xyz_D_run[:1]

,EPC,z_rel,polar,tagID,actual,ConfID,site,pgm,id,x,...,P,run,xy,z,xyz,Antenna,x_ant,y_ant,z_ant,D
0,BA0101000000000000000000,0.05,H,1,P,2026-02-03,HQ,AEP_OK,2026-02-04,1.82,...,31.5,2026-02-04 21:23:51,1.82__2.51,0.05,1.82__2.51__0.05,041A9F60,2.8,2.2,3.6,3.695808


In [54]:
Actuals_xyz_D_run['D'].describe()

count    43200.000000
mean         5.530139
std          2.397750
min          1.064519
25%          3.685054
50%          4.913695
75%          6.817732
max         12.559849
Name: D, dtype: float64

In [55]:
Actuals_xyz_D_run.shape

(43200, 22)

In [56]:
Actuals_xyz_D_run.columns

Index(['EPC', 'z_rel', 'polar', 'tagID', 'actual', 'ConfID', 'site', 'pgm',
       'id', 'x', 'y', 'z0', 'P', 'run', 'xy', 'z', 'xyz', 'Antenna', 'x_ant',
       'y_ant', 'z_ant', 'D'],
      dtype='str')

In [57]:
index=['EPC', 'run', 'x', 'y', 'z', 'xyz', 'polar']
Actuals_xyz_D_run = pd.pivot_table(Actuals_xyz_D_run, index=index, columns=['Antenna'], values=['D'])
Cols_D = ['_'.join([str(y) for y in x]) for x in Actuals_xyz_D_run.columns]
Actuals_xyz_D_run.columns = Cols_D
Actuals_xyz_D_run = Actuals_xyz_D_run.reset_index(drop=False)
Actuals_xyz_D_run

,EPC,run,x,y,z,xyz,polar,D_041A9F60,D_041A9F61,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4
0,BA0101000000000000000000,2026-02-04 21:23:51,1.82,2.51,0.05,1.82__2.51__0.05,H,3.695808,3.878273,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592
1,BA0101000000000000000000,2026-02-04 21:25:01,1.82,2.51,0.05,1.82__2.51__0.05,H,3.695808,3.878273,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592
2,BA0101000000000000000000,2026-02-04 21:25:58,1.82,2.51,0.05,1.82__2.51__0.05,H,3.695808,3.878273,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592
3,BA0101000000000000000000,2026-02-04 21:28:08,3.76,3.33,0.05,3.76__3.33__0.05,H,3.847207,3.734568,8.454407,8.594475,4.362912,6.124622,4.567822,6.153454
4,BA0101000000000000000000,2026-02-04 21:28:55,3.76,3.33,0.05,3.76__3.33__0.05,H,3.847207,3.734568,8.454407,8.594475,4.362912,6.124622,4.567822,6.153454
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,BA0610000000000000000000,2026-02-06 22:19:48,2.65,3.03,1.67,2.65__3.03__1.67,H,2.106253,2.078533,8.797517,8.997794,3.900808,6.307797,3.929924,6.237011
5396,BA0610000000000000000000,2026-02-06 22:20:30,2.65,3.03,1.67,2.65__3.03__1.67,H,2.106253,2.078533,8.797517,8.997794,3.900808,6.307797,3.929924,6.237011
5397,BA0610000000000000000000,2026-02-06 22:22:32,3.34,2.01,1.67,3.34__2.01__1.67,H,2.013107,2.708985,8.042176,8.523884,3.808228,5.899373,3.190078,5.530335
5398,BA0610000000000000000000,2026-02-06 22:23:22,3.34,2.01,1.67,3.34__2.01__1.67,H,2.013107,2.708985,8.042176,8.523884,3.808228,5.899373,3.190078,5.530335


In [58]:
Actuals_xyz_D_run['nearestAnt'] =  Actuals_xyz_D_run[Cols_D].idxmin(axis=1).apply(lambda x:x.split('D_')[1])
Actuals_xyz_D_run ['D_nearestAnt'] = Actuals_xyz_D_run [Cols_D].min(axis=1)
Actuals_xyz_D_run

,EPC,run,x,y,z,xyz,polar,D_041A9F60,D_041A9F61,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt
0,BA0101000000000000000000,2026-02-04 21:23:51,1.82,2.51,0.05,1.82__2.51__0.05,H,3.695808,3.878273,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592,041A9F60,3.695808
1,BA0101000000000000000000,2026-02-04 21:25:01,1.82,2.51,0.05,1.82__2.51__0.05,H,3.695808,3.878273,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592,041A9F60,3.695808
2,BA0101000000000000000000,2026-02-04 21:25:58,1.82,2.51,0.05,1.82__2.51__0.05,H,3.695808,3.878273,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592,041A9F60,3.695808
3,BA0101000000000000000000,2026-02-04 21:28:08,3.76,3.33,0.05,3.76__3.33__0.05,H,3.847207,3.734568,8.454407,8.594475,4.362912,6.124622,4.567822,6.153454,041A9F61,3.734568
4,BA0101000000000000000000,2026-02-04 21:28:55,3.76,3.33,0.05,3.76__3.33__0.05,H,3.847207,3.734568,8.454407,8.594475,4.362912,6.124622,4.567822,6.153454,041A9F61,3.734568
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5395,BA0610000000000000000000,2026-02-06 22:19:48,2.65,3.03,1.67,2.65__3.03__1.67,H,2.106253,2.078533,8.797517,8.997794,3.900808,6.307797,3.929924,6.237011,041A9F61,2.078533
5396,BA0610000000000000000000,2026-02-06 22:20:30,2.65,3.03,1.67,2.65__3.03__1.67,H,2.106253,2.078533,8.797517,8.997794,3.900808,6.307797,3.929924,6.237011,041A9F61,2.078533
5397,BA0610000000000000000000,2026-02-06 22:22:32,3.34,2.01,1.67,3.34__2.01__1.67,H,2.013107,2.708985,8.042176,8.523884,3.808228,5.899373,3.190078,5.530335,041A9F60,2.013107
5398,BA0610000000000000000000,2026-02-06 22:23:22,3.34,2.01,1.67,3.34__2.01__1.67,H,2.013107,2.708985,8.042176,8.523884,3.808228,5.899373,3.190078,5.530335,041A9F60,2.013107


Actuals_xyz_D

In [59]:
Actuals_xyz_D = Actuals_xyz_D_run.drop(columns='run').drop_duplicates().reset_index(drop=True)
Actuals_xyz_D

Actuals_xyz_D['D_nearestAnt'].describe()

count    1800.000000
mean        3.039003
std         0.824036
min         1.064519
25%         2.476505
50%         3.027796
75%         3.574216
max         5.375481
Name: D_nearestAnt, dtype: float64

In [60]:
tags.shape

(216956, 14)

In [61]:
tags = pd.merge(tags, Actuals_xyz_D_run, on=['run', 'EPC'], how='inner')
tags.shape

(115842, 29)

## antRound

In [62]:
tags = tags.sort_values('datestamp').reset_index(drop=True)
tags.head()

,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),...,D_041A9F60,D_041A9F61,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt
0,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,...,2.482761,2.747017,9.620088,9.948171,4.881403,7.269120,4.636604,7.072348,041A9F60,2.482761
1,2026-02-04 21:23:51,BA0604000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-76.42,0.022803,PORT_4__NONE,-75.39,...,2.186641,2.482620,9.530656,9.855019,4.716715,7.159567,4.462891,6.959698,041A9F60,2.186641
2,2026-02-04 21:23:51,BA0508000000000000000000,2026-02-04 21:23:50.648,041A9F60__NONE,31.5,PORT_3__NONE,-76.44,0.022699,PORT_4__NONE,-71.35,...,2.482761,2.747017,9.620088,9.948171,4.881403,7.269120,4.636604,7.072348,041A9F60,2.482761
3,2026-02-04 21:23:51,BA0209000000000000000000,2026-02-04 21:23:50.655,041A9F60__NONE,31.5,PORT_3__NONE,-81.21,0.007568,PORT_4__NONE,-78.42,...,3.389602,3.587673,9.941901,10.278589,5.453018,7.664685,5.235017,7.478329,041A9F60,3.389602
4,2026-02-04 21:23:51,BA0103000000000000000000,2026-02-04 21:23:50.656,041A9F60__NONE,31.5,PORT_3__NONE,-83.28,0.004699,PORT_4__NONE,-80.77,...,3.695808,3.878273,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592,041A9F60,3.695808


In [63]:
tags['Tx'].unique()

<StringArray>
['041A9F60__NONE', '041A9F61__NONE',   'PORT_1__NONE',   'PORT_3__NONE',
   'PORT_4__NONE',   'PORT_2__NONE', '0F173B13__NONE', '0F173B12__NONE']
Length: 8, dtype: str

In [64]:
Tx_sequence = {'041A9F60__NONE':1, '041A9F61__NONE':2, '0F173B12__NONE':3, '0F173B13__NONE':4, 'PORT_1__NONE':5, 'PORT_2__NONE':6, 'PORT_3__NONE':7, 'PORT_4__NONE':8  }
Tx_sequence

{'041A9F60__NONE': 1,
 '041A9F61__NONE': 2,
 '0F173B12__NONE': 3,
 '0F173B13__NONE': 4,
 'PORT_1__NONE': 5,
 'PORT_2__NONE': 6,
 'PORT_3__NONE': 7,
 'PORT_4__NONE': 8}

In [65]:
tags['Tx_sequence'] = tags['Tx'].map(Tx_sequence)
tags[:1]

,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),...,D_041A9F61,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt,Tx_sequence
0,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,...,2.747017,9.620088,9.948171,4.881403,7.26912,4.636604,7.072348,041A9F60,2.482761,1


In [66]:
tags = tags.copy()
tags['antRound'] = tags.groupby('run')['Tx_sequence'].transform(lambda s: s.diff().lt(0).cumsum())
tags.head()


,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),...,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt,Tx_sequence,antRound
0,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,...,9.620088,9.948171,4.881403,7.269120,4.636604,7.072348,041A9F60,2.482761,1,0
1,2026-02-04 21:23:51,BA0604000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-76.42,0.022803,PORT_4__NONE,-75.39,...,9.530656,9.855019,4.716715,7.159567,4.462891,6.959698,041A9F60,2.186641,1,0
2,2026-02-04 21:23:51,BA0508000000000000000000,2026-02-04 21:23:50.648,041A9F60__NONE,31.5,PORT_3__NONE,-76.44,0.022699,PORT_4__NONE,-71.35,...,9.620088,9.948171,4.881403,7.269120,4.636604,7.072348,041A9F60,2.482761,1,0
3,2026-02-04 21:23:51,BA0209000000000000000000,2026-02-04 21:23:50.655,041A9F60__NONE,31.5,PORT_3__NONE,-81.21,0.007568,PORT_4__NONE,-78.42,...,9.941901,10.278589,5.453018,7.664685,5.235017,7.478329,041A9F60,3.389602,1,0
4,2026-02-04 21:23:51,BA0103000000000000000000,2026-02-04 21:23:50.656,041A9F60__NONE,31.5,PORT_3__NONE,-83.28,0.004699,PORT_4__NONE,-80.77,...,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592,041A9F60,3.695808,1,0


In [67]:
tags.groupby('run')['antRound'].nunique().describe()

count    90.000000
mean     28.055556
std       4.203693
min      20.000000
25%      25.000000
50%      27.000000
75%      31.000000
max      37.000000
Name: antRound, dtype: float64

In [68]:
tags.groupby(['run', 'antRound']).apply(lambda x:(x['datestamp'].max()-x['datestamp'].min()).total_seconds()).describe()

count    2525.000000
mean        0.626863
std         0.211104
min         0.000000
25%         0.520000
50%         0.620000
75%         0.763000
max         1.218000
dtype: float64

## antRound duration = 0 ???

In [69]:
# tags = tags.drop(columns=['antRound_duration'])

In [70]:
antRounds = tags.groupby(['run', 'antRound']).apply(lambda x:(x['datestamp'].max()-x['datestamp'].min()).total_seconds()).rename('antRound_duration').reset_index(drop=False)
antRounds
# antRounds [antRounds['antRound_duration']==0]

,run,antRound,antRound_duration
0,2026-02-04 21:23:51,0,0.609
1,2026-02-04 21:23:51,1,0.799
2,2026-02-04 21:23:51,2,0.701
3,2026-02-04 21:23:51,3,0.659
4,2026-02-04 21:23:51,4,0.619
...,...,...,...
2520,2026-02-06 22:24:02,30,0.440
2521,2026-02-06 22:24:02,31,0.440
2522,2026-02-06 22:24:02,32,1.021
2523,2026-02-06 22:24:02,33,0.479


In [71]:
tags = pd.merge(tags, antRounds, on=['run', 'antRound'])

In [72]:
tags[:1]

,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),...,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt,Tx_sequence,antRound,antRound_duration
0,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,...,9.948171,4.881403,7.26912,4.636604,7.072348,041A9F60,2.482761,1,0,0.609


In [73]:
tags.shape

(115842, 32)

In [74]:
tags = tags [tags['antRound_duration']!=0]
tags.shape

(115829, 32)

antRound versus all run

In [75]:
tags.groupby(['run', 'xyz']) ['EPC'].nunique().describe()

count    540.000000
mean       9.355556
std        1.313363
min        3.000000
25%        9.000000
50%       10.000000
75%       10.000000
max       10.000000
Name: EPC, dtype: float64

In [76]:
tags.groupby(['run','antRound', 'xyz']) ['EPC'].nunique().describe()

count    14299.000000
mean         5.090776
std          2.444433
min          1.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         10.000000
Name: EPC, dtype: float64

run with 28 antRounds >> 1 antRound. **WHY SUCH A DIFFERENCE**

In [77]:
tags[(tags['polar']=='H')].groupby(['run', 'xyz']) ['EPC'].nunique().describe()

count    540.000000
mean       5.807407
std        0.551554
min        2.000000
25%        6.000000
50%        6.000000
75%        6.000000
max        6.000000
Name: EPC, dtype: float64

In [78]:
# inside antenna rectangle
x_ant_min = antConf['x_ant'].min()
x_ant_max = antConf['x_ant'].max()
y_ant_min = antConf['y_ant'].min()
y_ant_max = antConf['y_ant'].max()

# polar
polar = 'H'

# z#1m
z_max=1.5
z_min = 0.5

tags[(tags['polar']==polar) & (tags['x']>=x_ant_min) & (tags['x']<=x_ant_max) & (tags['y']>=y_ant_min) & (tags['y']<=y_ant_max) & (tags['txPower_dBm']==31.5) &\
     (tags['z']>=z_min) & (tags['z']<=z_max)]\
        .groupby(['run', 'xyz']) ['EPC'].nunique().describe()

count    32.0
mean      6.0
std       0.0
min       6.0
25%       6.0
50%       6.0
75%       6.0
max       6.0
Name: EPC, dtype: float64

In [79]:
tags[(tags['polar']==polar) & (tags['x']>=x_ant_min) & (tags['x']<=x_ant_max) & (tags['y']>=y_ant_min) & (tags['y']<=y_ant_max) & (tags['txPower_dBm']==31.5) &\
     (tags['z']>=z_min) & (tags['z']<=z_max)]\
        .groupby(['run','antRound', 'xyz']) ['EPC'].nunique().describe()

count    742.000000
mean       4.194070
std        1.410417
min        1.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        6.000000
Name: EPC, dtype: float64

1 antRound tells less than a run with >20antRounds

In [80]:
tags.groupby(['EPC', 'run', 'antRound']) ['Tx'].nunique().describe()

count    72793.000000
mean         1.591087
std          0.855607
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max          7.000000
Name: Tx, dtype: float64

In [81]:
tags.groupby(['EPC', 'run']) ['Tx'].nunique().describe()

count    5052.000000
mean        3.674584
std         1.815430
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max         8.000000
Name: Tx, dtype: float64

## Slots

In [82]:
freq='1000ms' # '750ms' '1000ms' '1500ms' '2000ms'

In [83]:
def func(df):
    Tmin = df['datestamp'].min()
    Tmax = df['datestamp'].max()
    Slots = pd.DataFrame({'slotStart':pd.date_range(start=Tmin, end=Tmax, freq=freq)})
#     Slots['slot_id']=range(len(Slots))
    return Slots

Slots = tags.groupby('run').apply(func, include_groups=False).reset_index(drop=False).rename(columns={'level_1':'slot_id'})
Slots['slot_id'] = Slots['slot_id'].astype('int32')
Slots.head()

,run,slot_id,slotStart
0,2026-02-04 21:23:51,0,2026-02-04 21:23:50.647
1,2026-02-04 21:23:51,1,2026-02-04 21:23:51.647
2,2026-02-04 21:23:51,2,2026-02-04 21:23:52.647
3,2026-02-04 21:23:51,3,2026-02-04 21:23:53.647
4,2026-02-04 21:23:51,4,2026-02-04 21:23:54.647


In [84]:
Slots.groupby('run').size().describe()

count    90.000000
mean     21.444444
std       1.162244
min      20.000000
25%      21.000000
50%      21.000000
75%      21.000000
max      28.000000
dtype: float64

In [85]:
Slots_min_max = pd.concat([\
           Slots.groupby('run')['slotStart'].min().rename('slotStart_min'),\
           Slots.groupby('run')['slotStart'].max().rename('slotStart_max')\
           ], axis=1).reset_index(drop=False)
Slots_min_max

,run,slotStart_min,slotStart_max
0,2026-02-04 21:23:51,2026-02-04 21:23:50.647,2026-02-04 21:24:10.647
1,2026-02-04 21:25:01,2026-02-04 21:25:01.076,2026-02-04 21:25:21.076
2,2026-02-04 21:25:58,2026-02-04 21:25:58.016,2026-02-04 21:26:18.016
3,2026-02-04 21:28:08,2026-02-04 21:28:07.774,2026-02-04 21:28:27.774
4,2026-02-04 21:28:55,2026-02-04 21:28:55.535,2026-02-04 21:29:15.535
...,...,...,...
85,2026-02-06 22:19:48,2026-02-06 22:19:47.814,2026-02-06 22:20:07.814
86,2026-02-06 22:20:30,2026-02-06 22:20:29.674,2026-02-06 22:20:50.674
87,2026-02-06 22:22:32,2026-02-06 22:22:32.374,2026-02-06 22:22:52.374
88,2026-02-06 22:23:22,2026-02-06 22:23:22.016,2026-02-06 22:23:42.016


sanitycheck

In [86]:
(Slots_min_max['slotStart_min'] > Slots_min_max['slotStart_max'].shift(1))[1:].mean()

np.float64(1.0)

In [87]:
Runs_size = Slots.groupby('run').size().rename('run_size').reset_index(drop=False)
Runs_size [Runs_size['run_size']>60]

,run,run_size


## merge_asof tags Slots

In [88]:
def func(run, x):
  # print(run)
  Slots_run = Slots [ Slots['run']==run ]
  x = x.copy()
  x['run'] = run
  x=x.sort_values('datestamp', ascending=True)
  Slots_run=Slots_run.sort_values('slotStart', ascending=True)
  x = pd.merge_asof(x, Slots_run.drop(columns=['run']) , left_on='datestamp', right_on='slotStart', direction='nearest')
  return x

tags = pd.concat([func(run, x) for run, x in tags.groupby('run')], ignore_index=True)
tags.head()


,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),...,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt,Tx_sequence,antRound,antRound_duration,slot_id,slotStart
0,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,...,7.269120,4.636604,7.072348,041A9F60,2.482761,1,0,0.609,0,2026-02-04 21:23:50.647
1,2026-02-04 21:23:51,BA0604000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-76.42,0.022803,PORT_4__NONE,-75.39,...,7.159567,4.462891,6.959698,041A9F60,2.186641,1,0,0.609,0,2026-02-04 21:23:50.647
2,2026-02-04 21:23:51,BA0508000000000000000000,2026-02-04 21:23:50.648,041A9F60__NONE,31.5,PORT_3__NONE,-76.44,0.022699,PORT_4__NONE,-71.35,...,7.269120,4.636604,7.072348,041A9F60,2.482761,1,0,0.609,0,2026-02-04 21:23:50.647
3,2026-02-04 21:23:51,BA0209000000000000000000,2026-02-04 21:23:50.655,041A9F60__NONE,31.5,PORT_3__NONE,-81.21,0.007568,PORT_4__NONE,-78.42,...,7.664685,5.235017,7.478329,041A9F60,3.389602,1,0,0.609,0,2026-02-04 21:23:50.647
4,2026-02-04 21:23:51,BA0103000000000000000000,2026-02-04 21:23:50.656,041A9F60__NONE,31.5,PORT_3__NONE,-83.28,0.004699,PORT_4__NONE,-80.77,...,7.817225,5.455914,7.634592,041A9F60,3.695808,1,0,0.609,0,2026-02-04 21:23:50.647


In [89]:
tags = tags.sort_values('datestamp').reset_index(drop=True)
tags.head()

,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),...,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt,Tx_sequence,antRound,antRound_duration,slot_id,slotStart
0,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,...,7.269120,4.636604,7.072348,041A9F60,2.482761,1,0,0.609,0,2026-02-04 21:23:50.647
1,2026-02-04 21:23:51,BA0604000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-76.42,0.022803,PORT_4__NONE,-75.39,...,7.159567,4.462891,6.959698,041A9F60,2.186641,1,0,0.609,0,2026-02-04 21:23:50.647
2,2026-02-04 21:23:51,BA0508000000000000000000,2026-02-04 21:23:50.648,041A9F60__NONE,31.5,PORT_3__NONE,-76.44,0.022699,PORT_4__NONE,-71.35,...,7.269120,4.636604,7.072348,041A9F60,2.482761,1,0,0.609,0,2026-02-04 21:23:50.647
3,2026-02-04 21:23:51,BA0209000000000000000000,2026-02-04 21:23:50.655,041A9F60__NONE,31.5,PORT_3__NONE,-81.21,0.007568,PORT_4__NONE,-78.42,...,7.664685,5.235017,7.478329,041A9F60,3.389602,1,0,0.609,0,2026-02-04 21:23:50.647
4,2026-02-04 21:23:51,BA0103000000000000000000,2026-02-04 21:23:50.656,041A9F60__NONE,31.5,PORT_3__NONE,-83.28,0.004699,PORT_4__NONE,-80.77,...,7.817225,5.455914,7.634592,041A9F60,3.695808,1,0,0.609,0,2026-02-04 21:23:50.647


In [90]:
tags.shape

(115829, 34)

# visu tags & sanity check statistics tags

In [91]:
# def visu(df, EPC='all', max_points_per_combo=800):
#     """
#     Visualize RSSI vs time for all (Tx,Rx) pairs.

#     EPC:  'all'  → all tags
#           <EPC string> → only that tag

#     max_points_per_combo: if not None, randomly subsample per (Tx,Rx)
#                           to reduce clutter.
#     """
#     import matplotlib.pyplot as plt

#     plt.rcParams.update({'axes.labelsize': 16,
#                          'xtick.labelsize': 14,
#                          'ytick.labelsize': 14})

#     # -------- Styles per (Tx,Rx) --------
#     # Colors are helpers; marker shapes carry the main information.
#     dict_styles = {
#         (1, 1): dict(marker='o',  color='tab:blue',    label='T1R1'),
#         (1, 2): dict(marker='s',  color='tab:orange',  label='T1R2'),
#         (1, 3): dict(marker='^',  color='tab:green',   label='T1R3'),
#         (1, 4): dict(marker='X',  color='tab:red',     label='T1R4'),

#         (2, 1): dict(marker='D',  color='tab:purple',  label='T2R1'),
#         (2, 2): dict(marker='v',  color='tab:brown',   label='T2R2'),
#         (2, 3): dict(marker='P',  color='tab:pink',    label='T2R3'),
#         (2, 4): dict(marker='*',  color='tab:gray',    label='T2R4'),

#         (3, 1): dict(marker='h',  color='tab:cyan',    label='T3R1'),
#         (3, 2): dict(marker='H',  color='tab:olive',   label='T3R2'),
#         (3, 3): dict(marker='<',  color='tab:blue',    label='T3R3'),
#         (3, 4): dict(marker='>',  color='tab:orange',  label='T3R4'),

#         (4, 1): dict(marker='8',  color='tab:green',   label='T4R1'),
#         (4, 2): dict(marker='p',  color='tab:red',     label='T4R2'),
#         (4, 3): dict(marker='x',  color='tab:purple',  label='T4R3'),
#         (4, 4): dict(marker='+',  color='tab:brown',   label='T4R4'),
#     }

#     # -------- Filter by EPC if needed --------
#     if EPC != 'all':
#         df = df[df['EPC'] == EPC]

#     if df.empty:
#         print("No data for this EPC.")
#         return

#     Tmax = df['datestamp'].max()
#     Tmin = df['datestamp'].min()
#     RSSImax = df['RSSI (dBm)'].max()
#     RSSImin = df['RSSI (dBm)'].min()

#     detections = len(df)
#     EPCs_unique = df['EPC'].nunique()

#     x_min = Tmin - pd.Timedelta(1, unit='s')
#     x_max = Tmax + pd.Timedelta(1, unit='s')

#     fig, ax = plt.subplots(figsize=(14, 6))

#     # -------- Plot each (Tx,Rx) --------
#     for (Tx, Rx), df_TxRx in df.groupby(['Tx', 'Rx']):
#         style = dict_styles.get(
#             (Tx, Rx),
#             dict(marker='o', color='black', label=f'T{Tx}R{Rx}')
#         )

#         # Subsample to avoid huge clouds
#         if max_points_per_combo is not None and len(df_TxRx) > max_points_per_combo:
#             df_plot = df_TxRx.sample(max_points_per_combo, random_state=0)
#         else:
#             df_plot = df_TxRx

#         ax.scatter(
#             df_plot['datestamp'],
#             df_plot['RSSI (dBm)'],
#             marker=style['marker'],
#             s=15,              # slightly smaller markers
#             alpha=0.6,
#             edgecolors='none',
#             color=style['color'],
#             label=style['label']
#         )

#     # -------- Titles & axes --------
#     title = f'EPC={EPC}, detections={detections}'
#     if EPC == 'all':
#         title += f', EPCs_unique={EPCs_unique}'
#     ax.set_title(title, size=16)

#     ax.set_xlim(x_min, x_max)
#     ax.set_ylim(RSSImin - 1, RSSImax + 1)
#     ax.set_ylabel('RSSI (dBm)')
#     ax.set_xlabel('datestamp')

#     # -------- Legend (unique labels) --------
#     handles, labels = ax.get_legend_handles_labels()
#     by_label = dict(zip(labels, handles))
#     ax.legend(by_label.values(),
#               by_label.keys(),
#               bbox_to_anchor=(1.02, 1),
#               loc='upper left',
#               fontsize=8,
#               title='Tx-Rx')

#     plt.tight_layout()
#     plt.show()


In [92]:
# for run in tags['run'].unique():
# # for run in tags['run'].unique():
#   # print(Runs[Runs['run']==run][ ['x', 'y', 'z']] )
#   # tags_run = tags [ (tags['run'] == run)    ]
#   tags_run = tags [ (tags['run'] == run)     ]

#   visu(tags_run, 'all')

In [93]:
# for run in pd.Series(tags['run'].unique()).sample(1):
# # for run in tags['run'].unique():
#   # print(Runs[Runs['run']==run][ ['x', 'y', 'z']] )
#   # tags_run = tags [ (tags['run'] == run)    ]
#   print(run)
#   tags_run = tags [ (tags['run'] == run)     ]
#   epc = pd.Series(tags_run['EPC'].unique()).sample(1).values[0]
#   print(epc)
#   visu(tags_run, epc)

#   # visu(tags_run, 'all')

In [94]:
# tags.groupby('Tx').size()

## nearest antenna = RSSImax?

In [95]:
# tags_TxRx = pd.melt(tags, id_vars=['run', 'EPC', 'datestamp', 'Tx', 'rx1_ant', 'rx2_ant'], value_vars=['rssi1 (dBm)', 'rssi2 (dBm)'], value_name='RSSI (dBm)', var_name='Rx').sort_values(['datestamp', 'EPC']) .reset_index(drop=True)
# tags_TxRx

In [96]:
# crit = tags_TxRx['Rx']=='rssi1 (dBm)'
# tags_TxRx['Rx'] = np.where(crit, tags_TxRx['rx1_ant'], tags_TxRx['rx2_ant'])
# tags_TxRx = tags_TxRx.drop(columns=['rx1_ant', 'rx2_ant'])
# tags_TxRx


RSSImax per (Tx, Rx)

In [97]:
# tags_TxRx_pick = tags_TxRx.copy()
# tags_TxRx_pick = tags_TxRx [ tags_TxRx['Tx'] == tags_TxRx['Rx'] ]


In [98]:
# RSSImax_TxRx = tags_TxRx_pick.groupby(['run', 'EPC', 'Tx', 'Rx'])['RSSI (dBm)'].max().rename('RSSI (dBm)_max').reset_index(drop=False)
# RSSImax_TxRx = pd.merge(RSSImax_TxRx, Actuals_antConf_xyz, on=['run', 'EPC'])
# RSSImax_TxRx

In [99]:
# RSSImax_TxRx[:10]

In [100]:
# antConf

RSSImax_Tx

In [101]:
# tags_TxRx[:1]

In [102]:
# tags_mono = tags_TxRx [tags_TxRx['Tx']==tags_TxRx['Rx']]
# tags_mono.head()

In [103]:
# RSSImax_Tx = tags_TxRx_pick.groupby(['run', 'EPC']).apply(lambda x:x.nlargest(1, 'RSSI (dBm)')).reset_index(drop=True).rename(columns={'RSSI (dBm)':'RSSI (dBm)_max'})
# RSSImax_Tx = pd.merge(RSSImax_Tx, Actuals_antConf_xyz, on=[ 'EPC'])
# RSSImax_Tx.head()

pick

In [104]:
# RSSImax_Tx_pick = RSSImax_Tx.copy()
# # RSSImax_Tx_pick = RSSImax_Tx [(abs(RSSImax_Tx['x'])<=1) & (abs(RSSImax_Tx['y'])<=1)]

In [105]:
# (RSSImax_Tx_pick['Tx']==RSSImax_Tx_pick['nearestAnt']).mean()

## TX antennas per EPC

In [106]:
# tags.groupby(['run', 'EPC']) ['Tx'].nunique().describe()

## Uzair

In [107]:

# # Sanity Check - EPC Completeness

# print("\n CHECKING EPC COMPLETENESS")
# epc_counts = tags.groupby('run')['EPC'].nunique().reset_index()
# expected_tags = 10

# # Identify runs with missing tags
# missing_mask = epc_counts['EPC'] < expected_tags
# bad_runs = epc_counts[missing_mask]

# print(f"Total Runs: {len(epc_counts)}")
# print(f"Runs with full {expected_tags} tags: {len(epc_counts) - len(bad_runs)}")
# if not bad_runs.empty:
#     print(f"WARNING: {len(bad_runs)} runs have < {expected_tags} unique EPCs.")
#     print(bad_runs.head()) # Printing first few bad runs
# else:
#     print("SUCCESS: All runs have complete EPC coverage.")


## UZAIR: analysis shoud be done by (run, EPC) using RSSImax_TxRx

In [108]:

# # RSSI Visualization (Boxplots)

# print("\n RSSI BOXPLOTS (Tx vs Rx per EPC)")
# import seaborn as sns
# import matplotlib.pyplot as plt


# all_epcs = tags['EPC'].unique()
# print(f"Generating plots for {len(all_epcs)} unique EPCs...")

# # Loop through EVERY EPC
# for epc in all_epcs:
#     # Filter data for this specific tag
#     subset = tags_TxRx[tags_TxRx['EPC'] == epc].copy()

#     # Skip if no data (just in case)
#     if subset.empty:
#         print(f"Skipping {epc} (No data)")
#         continue

#     # Create the Plot
#     # kind='box': Boxplot
#     # col='Tx': Creates a separate subplot for each Tx antenna automatically
#     # col_wrap=4: Keeps them in one row (since you have 4 Tx)
#     g = sns.catplot(
#         data=subset,
#         x='Rx',
#         y='RSSI (dBm)',
#         col='Tx',
#         kind='box',
#         col_wrap=4,
#         height=4,
#         aspect=0.8,
#         palette='viridis',
#         sharey=True
#     )

#     # Formatting
#     g.fig.suptitle(f"RSSI Distribution for EPC: {epc}", y=1.02, fontsize=16, fontweight='bold')
#     g.set_axis_labels("Rx Antenna", "RSSI (dBm)")

#     plt.show() # Render the plot
#     print(f"Displayed plot for {epc}")



# ds

In [109]:
tags[:1]

,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),...,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt,Tx_sequence,antRound,antRound_duration,slot_id,slotStart
0,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,...,7.26912,4.636604,7.072348,041A9F60,2.482761,1,0,0.609,0,2026-02-04 21:23:50.647


In [110]:
tags.shape

(115829, 34)

In [111]:
tags['xyz'].nunique(), Runs['run'].nunique(), Runs[['x', 'y']].apply(lambda x: '__'.join([str(y) for y in x]), axis=1).nunique()

(180, 90, 30)

In [112]:
tags[:1]

,run,EPC,datestamp,Tx,txPower_dBm,rx1_ant,rssi1 (dBm),rssi1,rx2_ant,rssi2 (dBm),...,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt,Tx_sequence,antRound,antRound_duration,slot_id,slotStart
0,2026-02-04 21:23:51,BA0507000000000000000000,2026-02-04 21:23:50.647,041A9F60__NONE,31.5,PORT_3__NONE,-77.97,0.015959,PORT_4__NONE,-83.32,...,7.26912,4.636604,7.072348,041A9F60,2.482761,1,0,0.609,0,2026-02-04 21:23:50.647


## basic ds

In [113]:
# PX
# aggfunc_list = [max, min, np.mean]
aggfunc_list = [max, min]
# aggfunc_list = [max]
# aggfunc_list = [np.mean]
# aggfunc_list = [max]

# index_pivot = ['EPC', 'x', 'y', 'z', 'xyz', 'polar']
index_pivot = ['EPC', 'xyz']
# index_pivot = ['EPC', 'xyz', 'txPower_dBm']
# index_pivot = ['EPC', 'xyz', 'antRound']
columns_pivot = ['txPower_dBm', 'Tx']
# columns_pivot = ['Tx']

# ds = pd.pivot_table(tags, index=index_pivot, columns=columns_pivot, \
#                               values=['rssi1', 'rssi2', 'Delta_rssi', 'rssi1 (dBm)', 'rssi2 (dBm)', 'Delta_rssi (dBm)'], aggfunc=aggfunc_list)
ds = pd.pivot_table(tags, index=index_pivot, columns=columns_pivot, \
                              values=['rssi1', 'rssi2', 'Delta_rssi'], aggfunc=aggfunc_list)

Xcols_ds = ['__'.join([str(y) for y in x]) for x in ds.columns]
ds.columns = Xcols_ds

for col in Xcols_ds:
  if 'dBm' in col:
    fill_value=-130
  else:
    fill_value=0
  ds[col] = ds[col].fillna(fill_value)

ds = ds.reset_index(drop=False)
ds

/var/folders/s4/sm8jpx2132x8t3tyxyd_xkfr0000gn/T/ipykernel_46201/2595145119.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ds = ds.reset_index(drop=False)


,EPC,xyz,max__Delta_rssi__24.0__041A9F60__NONE,max__Delta_rssi__24.0__041A9F61__NONE,max__Delta_rssi__24.0__0F173B12__NONE,max__Delta_rssi__24.0__0F173B13__NONE,max__Delta_rssi__24.0__PORT_1__NONE,max__Delta_rssi__24.0__PORT_2__NONE,max__Delta_rssi__24.0__PORT_3__NONE,max__Delta_rssi__24.0__PORT_4__NONE,...,min__rssi2__27.0__PORT_3__NONE,min__rssi2__27.0__PORT_4__NONE,min__rssi2__31.5__041A9F60__NONE,min__rssi2__31.5__041A9F61__NONE,min__rssi2__31.5__0F173B12__NONE,min__rssi2__31.5__0F173B13__NONE,min__rssi2__31.5__PORT_1__NONE,min__rssi2__31.5__PORT_2__NONE,min__rssi2__31.5__PORT_3__NONE,min__rssi2__31.5__PORT_4__NONE
0,BA0101000000000000000000,0.58__3.17__0.05,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.017589,0.000000,...,0.006324,0.000000,0.038905,0.054576,0.000000,0.000000,0.000000,0.000000,0.000351,0.000000
1,BA0101000000000000000000,1.82__2.51__0.05,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.013002,0.013002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,BA0101000000000000000000,10.72__3.5__0.05,0.0,0.0,0.000000,0.000000,0.000000,0.121253,0.000000,0.000000,...,0.015031,0.000000,0.000000,0.000000,0.088105,0.049431,0.000000,0.012445,0.013335,0.055719
3,BA0101000000000000000000,11.01__3.87__1.08,0.0,0.0,0.009052,-0.008822,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.005808,0.000000,0.048641,0.000000,0.088920,0.000000,0.000000
4,BA0101000000000000000000,11.23__3.36__0.05,0.0,0.0,0.028931,0.000000,0.000000,-0.033067,0.000000,0.030470,...,0.000000,0.009376,0.000000,0.000000,0.021627,0.010447,0.000000,0.054325,0.000000,0.012560
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,BA0610000000000000000000,8.52__1.93__2.46,0.0,0.0,0.793888,0.000000,0.026042,0.160711,0.000000,1.395986,...,0.000000,0.025119,0.353997,0.000000,0.009311,0.000000,0.077804,0.023714,0.176604,0.040832
1764,BA0610000000000000000000,8.56__4.52__2.45,0.0,0.0,0.521154,0.000000,-0.152743,0.168921,0.000000,1.570642,...,0.000000,0.006531,0.025645,0.000000,0.025942,0.018664,0.061376,0.007178,0.035563,0.009661
1765,BA0610000000000000000000,8.5__2.55__1.67,0.0,0.0,0.000000,0.000000,-0.004942,0.077492,0.000000,0.156192,...,0.000000,0.014894,0.000000,0.025586,0.000000,0.000000,0.089950,0.041020,0.081096,0.015740
1766,BA0610000000000000000000,8.87__4.79__1.67,0.0,0.0,0.116893,0.229069,0.000000,0.368379,0.000000,0.020715,...,0.000000,0.004764,0.000000,0.013062,0.003162,0.024044,0.000000,0.012618,0.025704,0.017989


## basic ds with model per Power

In [114]:
# PX
# aggfunc_list = [max, min, np.mean]
aggfunc_list = [max, min]
# aggfunc_list = [max]
# aggfunc_list = [np.mean]
# aggfunc_list = [max]

# index_pivot = ['EPC', 'x', 'y', 'z', 'xyz', 'polar']
# index_pivot = ['EPC', 'xyz']
index_pivot = ['EPC', 'xyz', 'txPower_dBm']
# index_pivot = ['EPC', 'xyz', 'antRound']
# columns_pivot = ['txPower_dBm', 'Tx']
columns_pivot = ['Tx']

# ds = pd.pivot_table(tags, index=index_pivot, columns=columns_pivot, \
#                               values=['rssi1', 'rssi2', 'Delta_rssi', 'rssi1 (dBm)', 'rssi2 (dBm)', 'Delta_rssi (dBm)'], aggfunc=aggfunc_list)
ds = pd.pivot_table(tags, index=index_pivot, columns=columns_pivot, \
                              values=['rssi1', 'rssi2', 'Delta_rssi'], aggfunc=aggfunc_list)

Xcols_ds = ['__'.join([str(y) for y in x]) for x in ds.columns]
ds.columns = Xcols_ds

for col in Xcols_ds:
  if 'dBm' in col:
    fill_value=-130
  else:
    fill_value=0
  ds[col] = ds[col].fillna(fill_value)

ds = ds.reset_index(drop=False)
ds

,EPC,xyz,txPower_dBm,max__Delta_rssi__041A9F60__NONE,max__Delta_rssi__041A9F61__NONE,max__Delta_rssi__0F173B12__NONE,max__Delta_rssi__0F173B13__NONE,max__Delta_rssi__PORT_1__NONE,max__Delta_rssi__PORT_2__NONE,max__Delta_rssi__PORT_3__NONE,...,min__rssi1__PORT_3__NONE,min__rssi1__PORT_4__NONE,min__rssi2__041A9F60__NONE,min__rssi2__041A9F61__NONE,min__rssi2__0F173B12__NONE,min__rssi2__0F173B13__NONE,min__rssi2__PORT_1__NONE,min__rssi2__PORT_2__NONE,min__rssi2__PORT_3__NONE,min__rssi2__PORT_4__NONE
0,BA0101000000000000000000,0.58__3.17__0.05,24.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.017589,...,0.027227,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.009638,0.000000
1,BA0101000000000000000000,0.58__3.17__0.05,27.0,-0.142279,0.000000,0.000000,0.000000,0.000000,0.000000,0.012254,...,0.009863,0.000000,0.154525,0.000000,0.000000,0.000000,0.000000,0.000000,0.006324,0.000000
2,BA0101000000000000000000,0.58__3.17__0.05,31.5,-0.016620,-0.023530,0.000000,0.000000,0.000000,0.000000,0.047019,...,0.045814,0.000000,0.038905,0.054576,0.000000,0.000000,0.000000,0.000000,0.000351,0.000000
3,BA0101000000000000000000,1.82__2.51__0.05,27.0,0.000000,-0.019587,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.026915,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,BA0101000000000000000000,1.82__2.51__0.05,31.5,0.015640,-0.009261,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.013002,0.013002,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5047,BA0610000000000000000000,8.87__4.79__1.67,27.0,0.000000,0.000000,0.127778,0.203231,0.000000,0.339173,0.000000,...,0.000000,0.018030,0.000000,0.000000,0.006457,0.015311,0.000000,0.002080,0.000000,0.004764
5048,BA0610000000000000000000,8.87__4.79__1.67,31.5,0.000000,0.149868,0.247812,0.264518,0.000000,0.348375,-0.011316,...,0.007464,0.013243,0.000000,0.013062,0.003162,0.024044,0.000000,0.012618,0.025704,0.017989
5049,BA0610000000000000000000,9.01__3.85__1.67,24.0,0.000000,0.000000,0.678925,0.000000,0.000000,0.547413,0.000000,...,0.000000,0.060395,0.000000,0.000000,0.009727,0.000000,0.000000,0.009931,0.000000,0.000148
5050,BA0610000000000000000000,9.01__3.85__1.67,27.0,0.000000,0.000000,0.317207,0.000000,0.000000,0.551672,0.000000,...,0.000000,0.073961,0.000000,0.000000,0.006683,0.000000,0.000000,0.001629,0.000000,0.000927


## advanced ds

In [115]:
# PX
# aggfunc_list = [max, min, np.mean]
aggfunc_list = [max, min]
index_pivot = ['EPC', 'xyz']

columns_pivot = ['txPower_dBm', 'Tx', 'rx1_ant']
# columns_pivot = ['Tx', 'rx1_ant']
Values = ['rssi1']
ds_Tx_rx1_ant = pd.pivot_table(tags, index=index_pivot, columns=columns_pivot, \
                              values=Values, aggfunc=aggfunc_list)

columns_pivot = ['txPower_dBm', 'Tx', 'rx2_ant']
# columns_pivot = ['Tx', 'rx2_ant']
Values = ['rssi2']
ds_Tx_rx2_ant = pd.pivot_table(tags, index=index_pivot, columns=columns_pivot, \
                              values=Values, aggfunc=aggfunc_list)

columns_pivot = ['txPower_dBm', 'Tx', 'rx1_rx2_ant']
# columns_pivot = ['Tx', 'rx1_rx2_ant']
Values = ['Delta_rssi']

ds_Tx_rx1_rx2_ant = pd.pivot_table(tags, index=index_pivot, columns=columns_pivot, \
                              values=Values, aggfunc=aggfunc_list)


ds = pd.concat([ds_Tx_rx1_ant, ds_Tx_rx2_ant, ds_Tx_rx1_rx2_ant], axis=1)

Xcols_ds = ['__'.join([str(y) for y in x]) for x in ds.columns]
ds.columns = Xcols_ds

for col in Xcols_ds:
  if 'dBm' in col:
    fill_value=-130
  else:
    fill_value=0
  ds[col] = ds[col].fillna(fill_value)

ds = ds.reset_index(drop=False)
ds

/var/folders/s4/sm8jpx2132x8t3tyxyd_xkfr0000gn/T/ipykernel_46201/437336889.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ds = ds.reset_index(drop=False)


,EPC,xyz,max__rssi1__24.0__041A9F60__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_4__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_4__NONE,...,min__Delta_rssi__31.5__PORT_1__NONE__PORT_1__NONE__PORT_4__NONE,min__Delta_rssi__31.5__PORT_2__NONE__PORT_2__NONE__PORT_1__NONE,min__Delta_rssi__31.5__PORT_2__NONE__PORT_2__NONE__PORT_3__NONE,min__Delta_rssi__31.5__PORT_2__NONE__PORT_2__NONE__PORT_4__NONE,min__Delta_rssi__31.5__PORT_3__NONE__PORT_3__NONE__PORT_1__NONE,min__Delta_rssi__31.5__PORT_3__NONE__PORT_3__NONE__PORT_2__NONE,min__Delta_rssi__31.5__PORT_3__NONE__PORT_3__NONE__PORT_4__NONE,min__Delta_rssi__31.5__PORT_4__NONE__PORT_4__NONE__PORT_1__NONE,min__Delta_rssi__31.5__PORT_4__NONE__PORT_4__NONE__PORT_2__NONE,min__Delta_rssi__31.5__PORT_4__NONE__PORT_4__NONE__PORT_3__NONE
0,BA0101000000000000000000,0.58__3.17__0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.047019,0.045463,0.000000,0.000000,0.000000,0.000000
1,BA0101000000000000000000,1.82__2.51__0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,BA0101000000000000000000,10.72__3.5__0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.034006,0.000000,-0.000493,0.008442,-0.221302,-0.003691,0.000000,-0.253567,-0.038017
3,BA0101000000000000000000,11.01__3.87__1.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,-0.073854,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,BA0101000000000000000000,11.23__3.36__0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,-0.110793,0.000000,0.000000,0.000000,0.002402,-0.028685,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,BA0610000000000000000000,8.52__1.93__2.46,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-3.323713,-0.074607,0.018707,-3.355540,-0.163238,-0.552513,-3.220555,0.584826,0.329704,1.256347
1764,BA0610000000000000000000,8.56__4.52__2.45,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-4.329341,0.125440,0.144485,-4.854464,-0.030961,-0.335357,0.000000,1.044276,0.317101,0.927901
1765,BA0610000000000000000000,8.5__2.55__1.67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.655708,0.000000,-0.014596,-0.734425,0.000000,-0.177458,-0.718732,0.108999,-0.175889,0.154547
1766,BA0610000000000000000000,8.87__4.79__1.67,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.348375,0.192502,0.093892,-0.011316,-0.570632,-0.063955,0.047751,-0.495869,0.005562


In [116]:
ds.shape, len(Xcols_ds)


((1768, 650), 648)

In [117]:
ds.columns[-20:]

Index(['min__Delta_rssi__31.5__0F173B13__NONE__PORT_2__NONE__PORT_3__NONE',
       'min__Delta_rssi__31.5__0F173B13__NONE__PORT_2__NONE__PORT_4__NONE',
       'min__Delta_rssi__31.5__0F173B13__NONE__PORT_3__NONE__PORT_1__NONE',
       'min__Delta_rssi__31.5__0F173B13__NONE__PORT_3__NONE__PORT_2__NONE',
       'min__Delta_rssi__31.5__0F173B13__NONE__PORT_3__NONE__PORT_4__NONE',
       'min__Delta_rssi__31.5__0F173B13__NONE__PORT_4__NONE__PORT_1__NONE',
       'min__Delta_rssi__31.5__0F173B13__NONE__PORT_4__NONE__PORT_2__NONE',
       'min__Delta_rssi__31.5__0F173B13__NONE__PORT_4__NONE__PORT_3__NONE',
       'min__Delta_rssi__31.5__PORT_1__NONE__PORT_1__NONE__PORT_2__NONE',
       'min__Delta_rssi__31.5__PORT_1__NONE__PORT_1__NONE__PORT_3__NONE',
       'min__Delta_rssi__31.5__PORT_1__NONE__PORT_1__NONE__PORT_4__NONE',
       'min__Delta_rssi__31.5__PORT_2__NONE__PORT_2__NONE__PORT_1__NONE',
       'min__Delta_rssi__31.5__PORT_2__NONE__PORT_2__NONE__PORT_3__NONE',
       'min__Delta_rss

In [118]:
Xcols_ds_lin = [x for x in Xcols_ds if 'dBm' not in x]
Xcols_ds_dBm = [x for x in Xcols_ds if 'dBm' in x]
len(Xcols_ds_lin)

648

In [119]:
ds.shape

(1768, 650)

In [120]:
ds = pd.merge(ds, Actuals_xyz_D, on=['EPC', 'xyz'])
ds.head()

,EPC,xyz,max__rssi1__24.0__041A9F60__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_4__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_4__NONE,...,D_041A9F60,D_041A9F61,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt
0,BA0101000000000000000000,0.58__3.17__0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.297883,4.182559,11.275629,11.468906,6.459551,8.828692,6.494906,8.781788,041A9F61,4.182559
1,BA0101000000000000000000,1.82__2.51__0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.695808,3.878273,10.066330,10.405143,5.665421,7.817225,5.455914,7.634592,041A9F60,3.695808
2,BA0101000000000000000000,10.72__3.5__0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.776041,8.775699,4.440822,4.306147,6.346093,4.454313,6.654089,4.707961,0F17B313,4.306147
3,BA0101000000000000000000,11.01__3.87__1.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,8.748909,8.683974,3.718790,3.228219,6.013768,3.775897,6.475291,4.230532,0F17B313,3.228219
4,BA0101000000000000000000,11.23__3.36__0.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,9.220249,9.249703,4.383035,4.298023,6.768530,4.740148,7.019615,4.941558,0F17B313,4.298023


In [121]:
ds.shape

(1768, 664)

## ds_shuffle

In [122]:
ds_shuffle = ds.sample(frac=1).reset_index(drop=True)
ds_shuffle.head()

,EPC,xyz,max__rssi1__24.0__041A9F60__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_4__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_4__NONE,...,D_041A9F60,D_041A9F61,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt
0,BA0504000000000000000000,12.73__3.05__2.14,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,10.072686,10.163415,2.942278,2.812294,7.308967,4.642521,7.467061,4.769591,0F17B313,2.812294
1,BA0205000000000000000000,1.82__2.51__0.37,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,3.389602,3.587673,9.941901,10.278589,5.453018,7.664685,5.235017,7.478329,041A9F60,3.389602
2,BA0303000000000000000000,6.16__4.08__0.7,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.820166,4.523273,6.375735,6.194191,3.233883,3.974167,4.027158,4.379269,PORT_1,3.233883
3,BA0102000000000000000000,8.52__1.93__0.8400000000000001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,6.356800,6.707227,4.102548,4.841374,4.647677,3.586210,4.235906,3.067067,PORT_4,3.067067
4,BA0104000000000000000000,4.42__2.39__0.7300000000000001,0.188799,0.020277,0.044771,0.016331,0.121899,0.027542,0.036983,0.026002,...,3.301121,3.630895,7.424918,7.829776,3.778809,5.355689,3.452159,5.096411,041A9F60,3.301121


# pick

In [123]:
tags_pick = tags.copy()
ds_pick = ds_shuffle.copy()
Xcols_ds_lin_pick = Xcols_ds_lin

pick values

In [124]:
tags['txPower_dBm'].unique()

array([31.5, 27. , 24. ])

In [125]:
tags['polar'].unique()

<StringArray>
['V', 'H']
Length: 2, dtype: str

In [126]:
x_ant_min = antConf['x_ant'].min()
x_ant_max = antConf['x_ant'].max()
y_ant_min = antConf['y_ant'].min()
y_ant_max = antConf['y_ant'].max()
(x_ant_min, x_ant_max), (y_ant_min, y_ant_max)

((np.float64(2.7), np.float64(11.2)), (np.float64(1.7), np.float64(4.0)))

In [127]:
(tags['x'].min(), tags['x'].max()), (tags['x'].min(), tags['x'].max()), (tags['z'].min(), tags['z'].max())

((np.float64(0.58), np.float64(14.53)),
 (np.float64(0.58), np.float64(14.53)),
 (np.float64(0.05), np.float64(2.94)))

Runs inside the antenna rectangle

In [128]:
Runs [ (Runs['x']>x_ant_min) & (Runs['x']<x_ant_max) & (Runs['y']>y_ant_min) & (Runs['y']<y_ant_max)].shape, Runs.shape

((36, 11), (90, 11))

In [129]:
# HQ

# Pmax_list=[24, 27, 31.5]
Pmax_list=[31.5]

# polar_list = ['H', 'V']
polar_list = ['H']

z_max=1.5
z_min = 0.5
# z_max=3
# z_min = 0

In [130]:
tags.columns

Index(['run', 'EPC', 'datestamp', 'Tx', 'txPower_dBm', 'rx1_ant',
       'rssi1 (dBm)', 'rssi1', 'rx2_ant', 'rssi2 (dBm)', 'rssi2',
       'rx1_rx2_ant', 'Delta_rssi', 'Delta_rssi (dBm)', 'x', 'y', 'z', 'xyz',
       'polar', 'D_041A9F60', 'D_041A9F61', 'D_0F17B312', 'D_0F17B313',
       'D_PORT_1', 'D_PORT_2', 'D_PORT_3', 'D_PORT_4', 'nearestAnt',
       'D_nearestAnt', 'Tx_sequence', 'antRound', 'antRound_duration',
       'slot_id', 'slotStart'],
      dtype='str')

In [131]:
# tags_pick = tags.copy()

# tags_pick =  tags [  tags['txPower_dBm'].isin(Pmax_list) & tags['polar'].isin(polar_list) & tags['polar'].isin(polar_list) ]

# tags_pick =  tags [  tags['txPower_dBm'].isin(Pmax_list)]
tags_pick =  tags [  tags['txPower_dBm'].isin(Pmax_list) & tags['polar'].isin(polar_list) & (tags['x']>=x_ant_min) & (tags['x']<=x_ant_max) \
                   & (tags['y']>=y_ant_min) & (tags['y']<=y_ant_max) & (tags['z']>=z_min) & (tags['z']<=z_max) ]
# tags_pick =  tags [  tags['txPower_dBm'].isin(Pmax_list) & tags['polar'].isin(polar_list) & (tags['z']>=z_min) & (tags['z']<=z_max) ]

tags.shape, tags_pick.shape

((115829, 34), (5857, 34))

In [132]:
tags_pick['xyz'].nunique()

32

## detection

In [133]:
Detections = tags_pick.groupby(['x', 'y', 'z', 'xyz']) ['EPC'].nunique().rename('EPCs').reset_index(drop=False)
Detections

,x,y,z,xyz,EPCs
0,2.88,2.90,0.70,2.88__2.9__0.7,6
1,2.88,2.90,1.02,2.88__2.9__1.02,6
2,2.88,2.90,1.34,2.88__2.9__1.34,6
3,3.34,2.01,0.70,3.34__2.01__0.7,6
4,3.34,2.01,1.02,3.34__2.01__1.02,6
5,3.34,2.01,1.34,3.34__2.01__1.34,6
6,3.36,3.56,1.18,3.36__3.56__1.18,6
7,3.36,3.56,1.50,3.36__3.56__1.5,6
8,3.76,3.33,0.70,3.76__3.33__0.7,6
9,3.76,3.33,1.02,3.76__3.33__1.02,6


In [134]:
Detections['EPCs'].describe()

count    32.0
mean      6.0
std       0.0
min       6.0
25%       6.0
50%       6.0
75%       6.0
max       6.0
Name: EPCs, dtype: float64

In [135]:
MisDetections = Detections [Detections['EPCs']!=10]
MisDetections

,x,y,z,xyz,EPCs
0,2.88,2.90,0.70,2.88__2.9__0.7,6
1,2.88,2.90,1.02,2.88__2.9__1.02,6
2,2.88,2.90,1.34,2.88__2.9__1.34,6
3,3.34,2.01,0.70,3.34__2.01__0.7,6
4,3.34,2.01,1.02,3.34__2.01__1.02,6
5,3.34,2.01,1.34,3.34__2.01__1.34,6
6,3.36,3.56,1.18,3.36__3.56__1.18,6
7,3.36,3.56,1.50,3.36__3.56__1.5,6
8,3.76,3.33,0.70,3.76__3.33__0.7,6
9,3.76,3.33,1.02,3.76__3.33__1.02,6


## ds_pick, Xcols_ds_pick

In [136]:
ds_shuffle[:1]

,EPC,xyz,max__rssi1__24.0__041A9F60__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_4__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_4__NONE,...,D_041A9F60,D_041A9F61,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt
0,BA0504000000000000000000,12.73__3.05__2.14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.072686,10.163415,2.942278,2.812294,7.308967,4.642521,7.467061,4.769591,0F17B313,2.812294


In [137]:
# HQ
# ds_pick = ds_shuffle.copy()

# ds_pick =  ds_shuffle [   ds_shuffle['polar'].isin(polar_list)]
ds_pick =  ds_shuffle [   ds_shuffle['polar'].isin(polar_list) & (ds_shuffle['x']>=x_ant_min) & (ds_shuffle['x']<=x_ant_max) \
                   & (ds_shuffle['y']>=y_ant_min) & (ds_shuffle['y']<=y_ant_max) & (ds_shuffle['z']>=z_min) & (ds_shuffle['z']<=z_max)]
# ds_pick =  ds_shuffle [   ds_shuffle['polar'].isin(polar_list) &  (ds_shuffle['z']>=z_min) & (ds_shuffle['z']<=z_max) ]


ds_shuffle.shape, ds_pick.shape

((1768, 664), (192, 664))

power pick

In [138]:
Xcols_ds_lin_pick = []
for P in Pmax_list:
  Xcols_ds_pick_lin_P = [x for x in Xcols_ds_lin if str(P) in x]
  Xcols_ds_lin_pick = Xcols_ds_lin_pick + Xcols_ds_pick_lin_P
len(Xcols_ds_lin_pick)

# # HQ
# Xcols_ds_lin_pick = Xcols_ds_lin

216

In [139]:
Xcols_ds_lin_pick

['max__rssi1__31.5__041A9F60__NONE__PORT_1__NONE',
 'max__rssi1__31.5__041A9F60__NONE__PORT_2__NONE',
 'max__rssi1__31.5__041A9F60__NONE__PORT_3__NONE',
 'max__rssi1__31.5__041A9F60__NONE__PORT_4__NONE',
 'max__rssi1__31.5__041A9F61__NONE__PORT_1__NONE',
 'max__rssi1__31.5__041A9F61__NONE__PORT_2__NONE',
 'max__rssi1__31.5__041A9F61__NONE__PORT_3__NONE',
 'max__rssi1__31.5__041A9F61__NONE__PORT_4__NONE',
 'max__rssi1__31.5__0F173B12__NONE__PORT_1__NONE',
 'max__rssi1__31.5__0F173B12__NONE__PORT_2__NONE',
 'max__rssi1__31.5__0F173B12__NONE__PORT_3__NONE',
 'max__rssi1__31.5__0F173B12__NONE__PORT_4__NONE',
 'max__rssi1__31.5__0F173B13__NONE__PORT_1__NONE',
 'max__rssi1__31.5__0F173B13__NONE__PORT_2__NONE',
 'max__rssi1__31.5__0F173B13__NONE__PORT_3__NONE',
 'max__rssi1__31.5__0F173B13__NONE__PORT_4__NONE',
 'max__rssi1__31.5__PORT_1__NONE__PORT_1__NONE',
 'max__rssi1__31.5__PORT_2__NONE__PORT_2__NONE',
 'max__rssi1__31.5__PORT_3__NONE__PORT_3__NONE',
 'max__rssi1__31.5__PORT_4__NONE__POR

## classification xyz (low interest)

In [140]:
# ds_pick/

In [141]:
# target = 'xyz'
# target_pred = 'xyz_pred'
# clf = RandomForestClassifier()
# for ds_test in np.array_split(ds_pick, 5):
#   idx_test = ds_test.index
#   idx_train = [x for x in ds_pick.index if x not in idx_test]
#   ds_train = ds_pick.loc[idx_train]
#   X_train = ds_train.loc[:, Xcols_ds_lin_pick]
#   X_test = ds_test.loc[:, Xcols_ds_lin_pick]
# #

#   y_train = ds_train.loc[:, target]
#   y_test = ds_test.loc[:, target]
#   clf.fit(X_train, y_train)
#   y_pred = clf.predict(X_test)

#   ds_pick.loc[idx_test, target_pred] = y_pred

# (ds_pick[target] == ds_pick[target_pred]).mean()

# **multiple Regressors**


In [142]:
# ds_pick = ds.copy()
# Xcols_ds_pick = Xcols_ds
# target='xyz'

In [143]:
# Xcols_ds_pick

In [144]:
# # Regression (RF, ET, GB, KNN, Ridge, DecisionTree, NN)

from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


X_df = ds_pick[Xcols_ds_lin_pick]
y_df = ds_pick[['x', 'y', 'z']]

X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(
    X_df, y_df,
    test_size=0.2,
    random_state=0
)

# Keep original row indices for downstream error analysis
idx_train = X_train_df.index.to_numpy()
idx_test = X_test_df.index.to_numpy()

X_train = X_train_df.to_numpy()
X_test = X_test_df.to_numpy()
y_train = y_train_df.to_numpy()
y_test = y_test_df.to_numpy()


# # Define candidate regressors

regressors = {
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=0),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=200, random_state=0),
    "GradientBoosting": GradientBoostingRegressor(random_state=0),
    "KNN": KNeighborsRegressor(n_neighbors=5),
    "Ridge": Ridge(alpha=1.0),
    "DecisionTree": DecisionTreeRegressor(random_state=0),
    "NN": MLPRegressor(
        hidden_layer_sizes=(64, 64),
        activation='relu',
        random_state=0,
        max_iter=1000,
        early_stopping=True
    ),
}

reg_results = {}    # RMSE per model
reg_errors = {}     # per-sample 3D distance error per model
y_pred_store = {}   # store predictions per model
best_model = None   # trained RandomForest model


for name, base_reg in regressors.items():
    model = MultiOutputRegressor(base_reg)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    # 3D RMSE on (x, y, z)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    reg_results[name] = rmse

    # per-sample Euclidean distance in 3D
    dists = np.linalg.norm(y_pred - y_test, axis=1)
    reg_errors[name] = dists

    # store predictions for later analysis (Task 3 & 4)
    y_pred_store[name] = y_pred

    print(
        f"{name}: 3D RMSE = {rmse:.3f} m, "
        f"mean |error| = {dists.mean():.3f} m, "
        f"median |error| = {np.median(dists):.3f} m"
    )

    # keep RandomForest as final model if it is good otherwise change it
    if name == "RandomForest":
         best_model = model

reg_results


RandomForest: 3D RMSE = 0.771 m, mean |error| = 1.089 m, median |error| = 0.955 m
ExtraTrees: 3D RMSE = 0.764 m, mean |error| = 1.045 m, median |error| = 0.877 m
GradientBoosting: 3D RMSE = 0.721 m, mean |error| = 1.071 m, median |error| = 1.007 m
KNN: 3D RMSE = 0.849 m, mean |error| = 1.064 m, median |error| = 0.842 m
Ridge: 3D RMSE = 1.435 m, mean |error| = 2.127 m, median |error| = 1.945 m
DecisionTree: 3D RMSE = 1.283 m, mean |error| = 1.510 m, median |error| = 0.986 m
NN: 3D RMSE = 1.458 m, mean |error| = 1.892 m, median |error| = 1.275 m


{'RandomForest': np.float64(0.7712701040613485),
 'ExtraTrees': np.float64(0.7644527709525902),
 'GradientBoosting': np.float64(0.7211343344571776),
 'KNN': np.float64(0.849013786847075),
 'Ridge': np.float64(1.4350610553006484),
 'DecisionTree': np.float64(1.2826125181811037),
 'NN': np.float64(1.4582544582392054)}

In [145]:
ds_pick

,EPC,xyz,max__rssi1__24.0__041A9F60__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F60__NONE__PORT_4__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_1__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_2__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_3__NONE,max__rssi1__24.0__041A9F61__NONE__PORT_4__NONE,...,D_041A9F60,D_041A9F61,D_0F17B312,D_0F17B313,D_PORT_1,D_PORT_2,D_PORT_3,D_PORT_4,nearestAnt,D_nearestAnt
3,BA0102000000000000000000,8.52__1.93__0.8400000000000001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,6.356800,6.707227,4.102548,4.841374,4.647677,3.586210,4.235906,3.067067,PORT_4,3.067067
4,BA0104000000000000000000,4.42__2.39__0.7300000000000001,0.188799,0.020277,0.044771,0.016331,0.121899,0.027542,0.036983,0.026002,...,3.301121,3.630895,7.424918,7.829776,3.778809,5.355689,3.452159,5.096411,041A9F60,3.301121
14,BA0109000000000000000000,3.36__3.56__1.18,0.051050,0.013305,0.068234,0.077983,0.043053,0.014454,0.073282,0.086896,...,2.831890,2.519841,8.387825,8.449473,3.614914,5.820275,3.984420,5.910296,041A9F61,2.519841
47,BA0209000000000000000000,11.01__3.87__1.4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,8.662159,8.596569,3.463957,2.909124,5.870520,3.543304,6.342476,4.024301,0F17B313,2.909124
55,BA0309000000000000000000,4.42__2.39__1.38,0.000000,0.000000,0.000000,0.000000,0.000000,0.001442,0.032810,0.009750,...,2.754796,3.142435,7.153384,7.555587,3.252830,4.998490,2.866862,4.719629,041A9F60,2.754796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1742,BA0109000000000000000000,8.52__1.93__0.8400000000000001,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,6.356800,6.707227,4.102548,4.841374,4.647677,3.586210,4.235906,3.067067,PORT_4,3.067067
1744,BA0502000000000000000000,9.01__3.85__1.34,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,6.811329,6.702701,4.024947,3.685132,4.187147,2.610785,4.775374,3.160411,PORT_2,2.610785
1746,BA0309000000000000000000,2.88__2.9__0.7,0.034435,0.090782,0.000000,0.018281,0.011722,0.014488,0.007674,0.013740,...,2.984359,3.041776,8.884503,9.131944,4.404816,6.529502,4.367883,6.430428,041A9F60,2.984359
1749,BA0109000000000000000000,11.01__3.87__1.08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,8.748909,8.683974,3.718790,3.228219,6.013768,3.775897,6.475291,4.230532,0F17B313,3.228219


AFTER REGRESSOR

In [146]:
# Compare both variants:
# 1) non-strict: xyz head sees [d_hat, h]
# 2) strict:     xyz head sees d_hat only

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# -----------------------------
# Dataset pick (advanced first)
# -----------------------------
USE_ADVANCED_DS = True

if USE_ADVANCED_DS and 'ds_pick' in globals() and 'Xcols_ds_lin_pick' in globals() and len(Xcols_ds_lin_pick) > 0:
    ds_model = ds_pick.copy()
    feat_cols = list(Xcols_ds_lin_pick)
    ds_mode = "advanced ds (ds_pick + Xcols_ds_lin_pick)"
elif 'ds_pick' in globals() and 'Xcols_ds_lin' in globals() and len(Xcols_ds_lin) > 0:
    ds_model = ds_pick.copy()
    feat_cols = list(Xcols_ds_lin)
    ds_mode = "ds_pick + Xcols_ds_lin"
elif 'ds' in globals() and 'Xcols_ds_lin' in globals() and len(Xcols_ds_lin) > 0:
    ds_model = ds.copy()
    feat_cols = list(Xcols_ds_lin)
    ds_mode = "normal ds + Xcols_ds_lin"
elif 'ds' in globals() and 'Xcols_ds' in globals() and len(Xcols_ds) > 0:
    ds_model = ds.copy()
    feat_cols = list(Xcols_ds)
    ds_mode = "normal ds + Xcols_ds"
else:
    raise ValueError("Need ds/ds_pick + Xcols_ds* variables.")

X_df = ds_model.loc[:, feat_cols].copy()
X_df = X_df.apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)
y_df = ds_model[['x', 'y', 'z']].astype('float32')

print(f"Using: {ds_mode}")
print(f"Samples={len(X_df)}, Features={X_df.shape[1]}")

# Fixed split for fair comparison
X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(
    X_df, y_df, test_size=0.2, random_state=0
)
idx_train = X_train_df.index.to_numpy()
idx_test = X_test_df.index.to_numpy()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df.to_numpy(dtype='float32')).astype('float32')
X_test  = scaler.transform(X_test_df.to_numpy(dtype='float32')).astype('float32')
y_train = y_train_df.to_numpy(dtype='float32')
y_test  = y_test_df.to_numpy(dtype='float32')

# Anchor geometry for distance supervision
if {'x_ant', 'y_ant', 'z_ant'}.issubset(antConf.columns):
    ant_xyz = antConf[['x_ant', 'y_ant', 'z_ant']].drop_duplicates().to_numpy(dtype='float32')
elif {'x', 'y', 'z'}.issubset(antConf.columns):
    ant_xyz = antConf[['x', 'y', 'z']].drop_duplicates().to_numpy(dtype='float32')
else:
    raise ValueError("antConf needs [x_ant,y_ant,z_ant] or [x,y,z].")

if ant_xyz.shape[0] < 3:
    raise ValueError("Need at least 3 anchors.")

def dist_to_anchors(xyz, anchors):
    return np.linalg.norm(xyz[:, None, :] - anchors[None, :, :], axis=2).astype('float32')

D_train = dist_to_anchors(y_train, ant_xyz)
n_anchors = ant_xyz.shape[0]

def build_model(strict=True):
    inp = keras.Input(shape=(X_train.shape[1],), name='X')

    h = layers.Dense(192, activation='swish', kernel_regularizer=regularizers.l2(2e-4))(inp)
    h = layers.BatchNormalization()(h)
    h = layers.Dropout(0.30)(h)
    h = layers.Dense(96, activation='swish', kernel_regularizer=regularizers.l2(2e-4))(h)
    h = layers.Dropout(0.20)(h)

    d_hat = layers.Dense(64, activation='swish', kernel_regularizer=regularizers.l2(1e-4))(h)
    d_hat = layers.Dropout(0.15)(d_hat)
    d_hat = layers.Dense(n_anchors, activation='softplus', name='d_hat')(d_hat)

    t = d_hat if strict else layers.Concatenate()([d_hat, h])

    t = layers.Dense(96, activation='swish', kernel_regularizer=regularizers.l2(1e-4))(t)
    t = layers.Dropout(0.20)(t)
    t = layers.Dense(48, activation='swish', kernel_regularizer=regularizers.l2(1e-4))(t)
    xyz = layers.Dense(3, name='xyz')(t)

    model = keras.Model(inp, [xyz, d_hat], name=f"KerasLearnedTrilat_{'strict' if strict else 'non_strict'}")
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss={'xyz': keras.losses.Huber(delta=1.0), 'd_hat': keras.losses.Huber(delta=0.5)},
        loss_weights={'xyz': 1.0, 'd_hat': 0.25},
        metrics={'xyz': [keras.metrics.MeanAbsoluteError(name='mae')]}
    )
    return model

def train_eval(strict, seed=42):
    tf.keras.utils.set_random_seed(seed)
    np.random.seed(seed)

    model = build_model(strict=strict)
    cbs = [
    callbacks.EarlyStopping(
        monitor='val_xyz_loss',
        mode='min',
        patience=35,
        restore_best_weights=True
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_xyz_loss',
        mode='min',
        factor=0.5,
        patience=12,
        min_lr=1e-5,
        verbose=0
    ),
    ]


    bs = int(min(256, max(32, len(X_train)//12)))
    hist = model.fit(
        X_train, {'xyz': y_train, 'd_hat': D_train},
        validation_split=0.2,
        epochs=500,
        batch_size=bs,
        callbacks=cbs,
        verbose=0
    )

    pred_xyz, pred_d = model.predict(X_test, verbose=0)
    dists = np.linalg.norm(pred_xyz - y_test, axis=1)
    rmse = np.sqrt(mean_squared_error(y_test, pred_xyz))

    # describe-style error spread
    err_desc = pd.Series(dists, name='err').describe()

    return model, pred_xyz, rmse, dists, err_desc, len(hist.history['loss'])

runs = {}
for strict_flag, name in [(False, "KerasLearnedTrilat_non_strict"), (True, "KerasLearnedTrilat_strict")]:
    model, pred_xyz, rmse, dists, err_desc, n_epochs = train_eval(strict=strict_flag, seed=42)

    runs[name] = {
        'model': model,
        'pred_xyz': pred_xyz,
        'rmse': rmse,
        'dists': dists,
        'err_desc': err_desc,
        'epochs': n_epochs
    }

    print(f"\n{name}")
    print(f"RMSE={rmse:.3f} | mean={dists.mean():.3f} | median={np.median(dists):.3f} | epochs={n_epochs}")
    print(err_desc)

# Notebook-compatible outputs
reg_results = {k: v['rmse'] for k, v in runs.items()}
reg_errors = {k: v['dists'] for k, v in runs.items()}
y_pred_store = {k: v['pred_xyz'] for k, v in runs.items()}
best_model = runs[min(reg_results, key=reg_results.get)]['model']  # best RMSE

summary = pd.DataFrame({
    'rmse': {k: v['rmse'] for k, v in runs.items()},
    'mean_err': {k: v['dists'].mean() for k, v in runs.items()},
    'median_err': {k: np.median(v['dists']) for k, v in runs.items()},
    'p90_err': {k: np.quantile(v['dists'], 0.90) for k, v in runs.items()},
    'p95_err': {k: np.quantile(v['dists'], 0.95) for k, v in runs.items()},
    'max_err': {k: v['dists'].max() for k, v in runs.items()},
    'epochs': {k: v['epochs'] for k, v in runs.items()},
}).sort_values('rmse')

print("\nComparison:")
display(summary)

reg_results


Using: advanced ds (ds_pick + Xcols_ds_lin_pick)
Samples=192, Features=216


2026-02-15 22:50:05.115993: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-02-15 22:50:05.116027: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-02-15 22:50:05.116038: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
I0000 00:00:1771192205.116048 7975588 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1771192205.116064 7975588 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-02-15 22:50:05.320055: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func;


KerasLearnedTrilat_non_strict
RMSE=0.869 | mean=1.181 | median=0.885 | epochs=205
count    39.000000
mean      1.180757
std       0.947008
min       0.236242
25%       0.610012
50%       0.884730
75%       1.400188
max       4.999333
Name: err, dtype: float64

KerasLearnedTrilat_strict
RMSE=0.779 | mean=1.058 | median=0.794 | epochs=149
count    39.000000
mean      1.058019
std       0.847189
min       0.312145
25%       0.564122
50%       0.793811
75%       1.290804
max       4.124890
Name: err, dtype: float64

Comparison:


2026-02-15 22:50:49.528320: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-02-15 22:50:49.528515: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

,rmse,mean_err,median_err,p90_err,p95_err,max_err,epochs
KerasLearnedTrilat_strict,0.778616,1.058019,0.793811,1.535736,2.807882,4.124890,149
KerasLearnedTrilat_non_strict,0.869485,1.180757,0.884730,2.061485,2.545845,4.999333,205


{'KerasLearnedTrilat_non_strict': np.float64(0.869485473798836),
 'KerasLearnedTrilat_strict': np.float64(0.7786161274596597)}

In [147]:
# Aggressively nonlinear xyz-only model (no distance head)

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.metrics import mean_squared_error

# expects: Xtr, Xte, ytr, yte already prepared (from your previous split/scaler cell)
# if your vars are X_train/X_test/y_train/y_test, map them:
if 'Xtr' not in globals():
    Xtr, Xte, ytr, yte = X_train, X_test, y_train, y_test

tf.keras.utils.set_random_seed(42)
np.random.seed(42)

wd = 2e-4

def res_block(x, units, drop=0.18):
    s = x
    x = layers.Dense(units, activation='gelu', kernel_regularizer=regularizers.l2(wd))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(drop)(x)
    x = layers.Dense(units, activation='gelu', kernel_regularizer=regularizers.l2(wd))(x)
    x = layers.BatchNormalization()(x)
    if s.shape[-1] != units:
        s = layers.Dense(units, kernel_regularizer=regularizers.l2(wd))(s)
    x = layers.Add()([x, s])
    x = layers.Activation('gelu')(x)
    return x

inp = keras.Input(shape=(Xtr.shape[1],), name='X')

# stronger nonlinearity on input space
x = layers.GaussianNoise(0.03)(inp)
x2 = layers.Lambda(lambda t: tf.square(t))(x)
xsin = layers.Lambda(lambda t: tf.sin(np.pi * t))(x)
xcos = layers.Lambda(lambda t: tf.cos(np.pi * t))(x)
x = layers.Concatenate()([x, x2, xsin, xcos])

x = layers.Dense(384, activation='gelu', kernel_regularizer=regularizers.l2(wd))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.22)(x)

x = res_block(x, 384, drop=0.20)
x = res_block(x, 256, drop=0.18)
x = res_block(x, 128, drop=0.15)

x = layers.Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(wd))(x)
x = layers.Dropout(0.12)(x)
out = layers.Dense(3, name='xyz')(x)

model_xyz_nl = keras.Model(inp, out, name='KerasXYZOnly_AggressiveNL')
model_xyz_nl.compile(
    optimizer=keras.optimizers.Adam(learning_rate=8e-4, clipnorm=1.0),
    loss=keras.losses.Huber(delta=1.0),
    metrics=[keras.metrics.MeanAbsoluteError(name='mae')]
)

cbs = [
    callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=60, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', mode='min', factor=0.5, patience=18, min_lr=1e-5, verbose=0),
]

bs = int(min(128, max(16, len(Xtr)//10)))
hist = model_xyz_nl.fit(
    Xtr, ytr,
    validation_split=0.25,
    epochs=900,
    batch_size=bs,
    callbacks=cbs,
    verbose=0
)

pred_xyz_nl = model_xyz_nl.predict(Xte, verbose=0)
err_nl = np.linalg.norm(pred_xyz_nl - yte, axis=1)
rmse_nl = np.sqrt(mean_squared_error(yte, pred_xyz_nl))

print(f"KerasXYZOnly_AggressiveNL: RMSE={rmse_nl:.3f} | mean={err_nl.mean():.3f} | median={np.median(err_nl):.3f} | epochs={len(hist.history['loss'])}")
print(pd.Series(err_nl, name='err').describe())


2026-02-15 22:50:55.141263: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-02-15 22:50:55.141485: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

KerasXYZOnly_AggressiveNL: RMSE=0.685 | mean=1.083 | median=0.989 | epochs=350
count    39.000000
mean      1.083449
std       0.490814
min       0.310686
25%       0.689922
50%       0.989480
75%       1.458510
max       2.038375
Name: err, dtype: float64


In [148]:
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.multioutput import MultiOutputRegressor

print("Current reg_errors keys:", list(reg_errors.keys()) if 'reg_errors' in globals() else "reg_errors not defined")

# Reuse same split if available
if all(v in globals() for v in ["X_df", "y_df", "idx_train", "idx_test"]):
    X_train_et = X_df.loc[idx_train].to_numpy()
    X_test_et  = X_df.loc[idx_test].to_numpy()
    y_train_et = y_df.loc[idx_train].to_numpy()
    y_test_et  = y_df.loc[idx_test].to_numpy()
else:
    # fallback: build from ds_pick / ds
    if 'ds_pick' in globals() and 'Xcols_ds_lin_pick' in globals() and len(Xcols_ds_lin_pick) > 0:
        ds_tmp = ds_pick.copy()
        feat_cols = list(Xcols_ds_lin_pick)
    elif 'ds' in globals() and 'Xcols_ds_lin' in globals() and len(Xcols_ds_lin) > 0:
        ds_tmp = ds.copy()
        feat_cols = list(Xcols_ds_lin)
    else:
        raise ValueError("Need X_df/y_df+split vars or ds/ds_pick + feature columns.")

    from sklearn.model_selection import train_test_split
    X_tmp = ds_tmp[feat_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y_tmp = ds_tmp[['x', 'y', 'z']].astype('float32')
    X_train_et, X_test_et, y_train_et, y_test_et = train_test_split(
        X_tmp.to_numpy(), y_tmp.to_numpy(), test_size=0.2, random_state=0
    )

# Train ExtraTrees
et = MultiOutputRegressor(
    ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1)
)
et.fit(X_train_et, y_train_et)
y_pred_et = et.predict(X_test_et)

# 3D Euclidean error + describe
err_et = np.linalg.norm(y_pred_et - y_test_et, axis=1)
print(pd.Series(err_et, name="err").describe())

# Optional: store back in your dicts
if 'reg_errors' not in globals():
    reg_errors = {}
if 'y_pred_store' not in globals():
    y_pred_store = {}
reg_errors["ExtraTrees"] = err_et
y_pred_store["ExtraTrees"] = y_pred_et


Current reg_errors keys: ['KerasLearnedTrilat_non_strict', 'KerasLearnedTrilat_strict']
count    39.000000
mean      1.043674
std       0.833347
min       0.218878
25%       0.477500
50%       0.882217
75%       1.282916
max       4.221095
Name: err, dtype: float64


In [156]:
# Confidence model: estimate risk that 3D error exceeds 2 m.
from sklearn.model_selection import KFold
from sklearn.ensemble import ExtraTreesRegressor, ExtraTreesClassifier
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report, confusion_matrix

ERR_THRESHOLD = 1.0
N_SPLITS = 5
SEED = 42

# Reuse matrices from previous cell if available
X_train_c = X_train_et if 'X_train_et' in globals() else X_train
X_test_c = X_test_et if 'X_test_et' in globals() else X_test
y_train_c = y_train_et if 'y_train_et' in globals() else y_train
y_test_c = y_test_et if 'y_test_et' in globals() else y_test

# 1) Build out-of-fold errors (no leakage)
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_pred = np.zeros_like(y_train_c, dtype=float)

for tr_idx, va_idx in kf.split(X_train_c):
    reg_fold = MultiOutputRegressor(
        ExtraTreesRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
    )
    reg_fold.fit(X_train_c[tr_idx], y_train_c[tr_idx])
    oof_pred[va_idx] = reg_fold.predict(X_train_c[va_idx])

oof_err = np.linalg.norm(oof_pred - y_train_c, axis=1)
y_risk_train = (oof_err > ERR_THRESHOLD).astype(int)
print(f"Train OOF positives (> {ERR_THRESHOLD:.1f}m): {y_risk_train.sum()}/{len(y_risk_train)}")

# 2) Final xyz regressor + risk classifier
reg_final = MultiOutputRegressor(
    ExtraTreesRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)
)
reg_final.fit(X_train_c, y_train_c)
test_pred = reg_final.predict(X_test_c)
test_err = np.linalg.norm(test_pred - y_test_c, axis=1)
y_risk_test = (test_err > ERR_THRESHOLD).astype(int)

risk_clf = ExtraTreesClassifier(
    n_estimators=500,
    random_state=SEED,
    n_jobs=-1,
    class_weight='balanced_subsample',
    min_samples_leaf=2
)
risk_clf.fit(X_train_c, y_risk_train)
risk_proba_test = risk_clf.predict_proba(X_test_c)[:, 1]
confidence_test = 1.0 - risk_proba_test

# Store for downstream analysis
risk_store = {
    'threshold_m': ERR_THRESHOLD,
    'risk_proba_test': risk_proba_test,
    'confidence_test': confidence_test,
    'y_risk_test': y_risk_test,
    'test_err': test_err,
}

print('Risk proba summary:')
display(pd.Series(risk_proba_test, name='risk_proba').describe())
print('Confidence summary:')
display(pd.Series(confidence_test, name='confidence').describe())

if len(np.unique(y_risk_test)) == 2:
    print('PR-AUC:', round(average_precision_score(y_risk_test, risk_proba_test), 4))
    print('ROC-AUC:', round(roc_auc_score(y_risk_test, risk_proba_test), 4))
else:
    print('Test set has a single class for err>2m; ROC/PR AUC not defined.')

pred_label_50 = (risk_proba_test >= 0.5).astype(int)
print('Confusion matrix @0.5:')
print(confusion_matrix(y_risk_test, pred_label_50))
print(classification_report(y_risk_test, pred_label_50, digits=3, zero_division=0))

df_risk = pd.DataFrame({
    'risk_proba': risk_proba_test,
    'confidence': confidence_test,
    'err_true_m': test_err,
    'is_bad_gt2m': y_risk_test,
}).sort_values('risk_proba', ascending=False).reset_index(drop=True)
display(df_risk.head(20))


Train OOF positives (> 1.0m): 48/153
Risk proba summary:


count    39.000000
mean      0.396470
std       0.173097
min       0.036553
25%       0.285076
50%       0.423471
75%       0.525740
max       0.649598
Name: risk_proba, dtype: float64

Confidence summary:


count    39.000000
mean      0.603530
std       0.173097
min       0.350402
25%       0.474260
50%       0.576529
75%       0.714924
max       0.963447
Name: confidence, dtype: float64

PR-AUC: 0.5025
ROC-AUC: 0.625
Confusion matrix @0.5:
[[16  8]
 [ 7  8]]
              precision    recall  f1-score   support

           0      0.696     0.667     0.681        24
           1      0.500     0.533     0.516        15

    accuracy                          0.615        39
   macro avg      0.598     0.600     0.598        39
weighted avg      0.620     0.615     0.617        39



,risk_proba,confidence,err_true_m,is_bad_gt2m
0,0.649598,0.350402,2.511950,1
1,0.629454,0.370546,0.541037,0
2,0.625232,0.374768,0.489609,0
3,0.608349,0.391651,0.303470,0
4,0.606380,0.393620,1.714102,1
5,0.604332,0.395668,1.462852,1
6,0.600296,0.399704,0.968158,0
7,0.534092,0.465908,0.225053,0
8,0.533894,0.466106,0.413410,0
9,0.526157,0.473843,1.626681,1


In [158]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, recall_score, precision_score

# y_risk_test: 1 si err_true_m > 1m
# risk_proba_test: proba prédite de "bad"
y = y_risk_test.astype(int)
p = risk_proba_test

target_recall = 0.80
taus = np.unique(np.r_[np.linspace(0, 1, 1001), p])  # grille fine + proba observées

rows = []
for tau in taus:
    yhat = (p >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0,1]).ravel()
    recall_bad = tp / (tp + fn) if (tp + fn) else 0.0
    precision_bad = tp / (tp + fp) if (tp + fp) else 0.0
    flagged = tp + fp
    rows.append({
        "tau": tau, "tp": tp, "fp": fp, "fn": fn, "tn": tn,
        "recall_bad": recall_bad, "precision_bad": precision_bad,
        "flagged_total": flagged,
        "fp_rate_among_flagged": (fp / flagged) if flagged else 0.0,   # alertes inutiles / alertes
        "fp_rate_among_good": (fp / (fp + tn)) if (fp + tn) else 0.0   # bons points rejetés
    })

res = pd.DataFrame(rows)

# Parmi les taus qui atteignent recall >= 0.80, on prend celui qui minimise FP
cand = res[res["recall_bad"] >= target_recall].copy()

if cand.empty:
    print("Aucun tau n'atteint recall_bad >= 0.80 sur ce test.")
    best = res.sort_values(["recall_bad", "fp"], ascending=[False, True]).iloc[0]
else:
    best = cand.sort_values(["fp", "flagged_total", "tau"], ascending=[True, True, False]).iloc[0]

print("=== Seuil choisi ===")
print(best[["tau","recall_bad","precision_bad","tp","fn","fp","tn","flagged_total",
            "fp_rate_among_flagged","fp_rate_among_good"]])

# Optionnel: voir les faux positifs (tags flaggés à tort)
tau_star = float(best["tau"])
yhat_star = (p >= tau_star).astype(int)
fp_idx = np.where((yhat_star == 1) & (y == 0))[0]

print(f"\nFaux positifs: {len(fp_idx)}")
display(pd.DataFrame({
    "idx_test": fp_idx,
    "risk_proba": p[fp_idx],
    "err_true_m": risk_store["test_err"][fp_idx] if "risk_store" in globals() else np.nan
}).sort_values("risk_proba", ascending=False))


=== Seuil choisi ===
tau                       0.323616
recall_bad                0.800000
precision_bad             0.461538
tp                       12.000000
fn                        3.000000
fp                       14.000000
tn                       10.000000
flagged_total            26.000000
fp_rate_among_flagged     0.538462
fp_rate_among_good        0.583333
Name: 337, dtype: float64

Faux positifs: 14


,idx_test,risk_proba,err_true_m
4,8,0.629454,0.541037
1,1,0.625232,0.489609
10,19,0.608349,0.303470
2,3,0.600296,0.968158
6,12,0.534092,0.225053
8,16,0.533894,0.413410
3,7,0.525324,0.825199
11,20,0.512469,0.716802
5,10,0.488815,0.899068
12,30,0.336124,0.460387


In [160]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Ajuste selon ton notebook
# y_true: 1 si err_true_m > threshold, sinon 0
# y_pred: 1 si flag (risk_proba >= tau), sinon 0
y_true = (risk_store["test_err"] > ERR_THRESHOLD).astype(int)
y_pred = (risk_proba_test >= tau_star).astype(int)

tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

# 1) Sur 100 tags flaggés: combien correct / à tort
pct_correct_among_flagged = 100 * tp / (tp + fp) if (tp + fp) else 0
pct_wrong_among_flagged   = 100 * fp / (tp + fp) if (tp + fp) else 0

# 2) Sur 100 tags trop loin: combien flaggés / non flaggés
pct_flagged_among_bad     = 100 * tp / (tp + fn) if (tp + fn) else 0
pct_missed_among_bad      = 100 * fn / (tp + fn) if (tp + fn) else 0

print(f"Sur 100 tags flaggés: {pct_correct_among_flagged:.1f} corrects, {pct_wrong_among_flagged:.1f} à tort")
print(f"Sur 100 tags trop loin: {pct_flagged_among_bad:.1f} flaggés, {pct_missed_among_bad:.1f} non flaggés")


Sur 100 tags flaggés: 46.2 corrects, 53.8 à tort
Sur 100 tags trop loin: 80.0 flaggés, 20.0 non flaggés


In [161]:
# y_true: 1 si tag trop loin (err > threshold), sinon 0
# y_pred: 1 si tag flaggé, sinon 0
y_true = (risk_store["test_err"] > ERR_THRESHOLD).astype(int)
y_pred = (risk_proba_test >= tau_star).astype(int)

n = len(y_true)
n_flagged = y_pred.sum()
n_trop_loin = y_true.sum()

print(f"Tags flaggés: {n_flagged}/{n} ({100*n_flagged/n:.1f}%)")
print(f"Tags trop loin: {n_trop_loin}/{n} ({100*n_trop_loin/n:.1f}%)")


Tags flaggés: 26/39 (66.7%)
Tags trop loin: 15/39 (38.5%)


In [ ]:
# antConf

In [ ]:
# import numpy as np

# # Antennas for the 2m square (case 3)
# antennas_case1 = np.array([
#     # for case 2 and 3
#     # [ 0.0, -1.0, 0.0],  # South
#     # [ 1.0,  0.0, 0.0],  # East
#     # [ 0.0,  1.0, 0.0],  # North
#     # [-1.0,  0.0, 0.0],  # West
#     # for case 1
#     [-1.0, -1.0, 0.0],   # A1
#     [ 1.0, -1.0, 0.0],   # A2
#     [ 1.0,  1.0, 0.0],   # A3
#     [-1.0,  1.0, 0.0],   # A4
# ])

# def nearest_antenna(points, antennas):
#     d = np.linalg.norm(points[:,None,:] - antennas[None,:,:], axis=2)
#     return np.argmin(d, axis=1)

# # TRUE nearest antennas
# true_nearest = nearest_antenna(y_test, antennas_case1)

# # PRED nearest antennas for RandomForest only
# rf_pred = y_pred_store["RandomForest"]
# pred_nearest = nearest_antenna(rf_pred, antennas_case1)

# # Zone accuracy
# zone_accuracy = np.mean(true_nearest == pred_nearest)

# print(f"RF – Antenna Zone Accuracy: {zone_accuracy*100:.2f}%")


In [ ]:
# def build_error_df(model_name="RandomForest"):
#     """
#     Returns a DataFrame with, for each test sample:
#     - original ds index
#     - true (x,y,z)
#     - predicted (x,y,z)
#     - 3D error (Euclidean distance)
#     """
#     if model_name not in y_pred_store:
#         raise ValueError(
#             f"Unknown model '{model_name}'. Available: {list(y_pred_store.keys())}"
#         )

#     y_pred = y_pred_store[model_name]
#     dists = reg_errors[model_name]

#     # Build DataFrame
#     df_err = pd.DataFrame({
#         "ds_index": idx_test,
#         "x_true": y_test[:, 0],
#         "y_true": y_test[:, 1],
#         "z_true": y_test[:, 2],
#         "x_pred": y_pred[:, 0],
#         "y_pred": y_pred[:, 1],
#         "z_pred": y_pred[:, 2],
#         "error_3D": dists,
#     })

#     # Sort by descending error
#     df_err = df_err.sort_values(
#         by="error_3D", ascending=False
#     ).reset_index(drop=True)

#     return df_err


In [ ]:
# df_err_nn = build_error_df("NN")
# df_err_nn.head()

In [ ]:
# df_err_rf = build_error_df("RandomForest")
# df_err_nn.head()

In [ ]:
# def show_worst_samples(model_name="RandomForest", k=5):
#     df_err = build_error_df(model_name)
#     worst = df_err.head(k)

#     print(f"Worst {k} samples for model: {model_name}")
#     display(worst)

#     cols_to_show = ["EPC", "run"] if all(c in ds.columns for c in ["EPC", "run"]) else []
#     if cols_to_show:
#         display(
#             worst.merge(
#                 ds[cols_to_show],
#                 left_on="ds_index",
#                 right_index=True,
#                 how="left"
#             )
#         )

# show_worst_samples("NN", k=5)


In [ ]:
from mpl_toolkits.mplot3d import Axes3D


In [ ]:
# import numpy as np

# def cell_accuracy(y_true, y_pred, cell_size):
#     y_true = np.asarray(y_true)
#     y_pred = np.asarray(y_pred)

#     true_snapped = np.round(y_true / cell_size) * cell_size
#     pred_snapped = np.round(y_pred / cell_size) * cell_size

#     same_cell = np.all(true_snapped == pred_snapped, axis=1)
#     return same_cell.mean()

# def report_cell_accuracies(model_name="RandomForest", cell_sizes=(2.0, 1.0, 0.5)):
#     y_pred = y_pred_store[model_name]
#     print(f"Cell accuracy for model: {model_name}")
#     for G in cell_sizes:
#         acc = cell_accuracy(y_test, y_pred, G)
#         print(f"  grid {G:.2f} m: {acc*100:.1f}% of test points in correct cell")

# report_cell_accuracies("RandomForest", cell_sizes=(2.0, 1.0, 0.5))
# report_cell_accuracies("NN", cell_sizes=(2.0, 1.0, 0.5))


In [ ]:
# radii = [0.5, 1.0, 1.5, 2.0]

# for name, dists in reg_errors.items():
#     print(f"\n{name}:")
#     for R in radii:
#         frac = (dists <= R).mean()
#         print(f"  within {R:.1f} m: {frac*100:.1f}% of test points")


## tree

In [ ]:
# from sklearn.tree import DecisionTreeClassifier

In [ ]:
# X = ds_pick[Xcols_ds_pick]
# y = ds_pick[['x', 'y', 'z']].values

In [ ]:
# clf = DecisionTreeRegressor()
# clf.fit(X, y)
# y_pred =  clf.predict(X)
# np.sqrt(pd.Series([(x[0]**2+x[1]**2+x[2]**2)  for x in (y-y_pred)]).mean())

In [ ]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8)
# clf.fit(X_train, y_train)
# y_pred =  clf.predict(X_test)
# y_pred[:3]

In [ ]:
np.sqrt(pd.Series([(x[0]**2+x[1]**2+x[2]**2)  for x in (y_test-y_pred)]).mean())

# Best clf so far

In [ ]:
ds_pick.shape, len(Xcols_ds_lin_pick)

In [ ]:
print(ds_pick.columns)

In [ ]:
ds_pick.head()

In [ ]:
ds_pick['D_nearestAnt'].mean()

In [ ]:
Values_xyz = ['x', 'y', 'z']
Values_xyz_pred = [x+'_pred' for x in Values_xyz]

Values_D = Cols_D
Values_D_pred = [x+'_pred' for x in Values_D]

In [ ]:
Values_pick = Values_xyz
# Values_pick = Values_D
# Values_pick = Values_xyz + Values_D
Values_pick_pred = [x+'_pred' for x in Values_pick]

In [ ]:
Values_pick

In [ ]:
Values_pick_pred

## train_test_split, Kfold=5

In [ ]:
# clf = RandomForestRegressor()
# clf = KNeighborsRegressor(n_neighbors=5)
clf = ExtraTreesRegressor()
target = Values_pick
target_pred = Values_pick_pred
for idx_test in np.array_split(ds_pick.index.to_numpy(), 5):
  ds_test = ds_pick.loc[idx_test]
  idx_train = ds_pick.index.difference(idx_test)
  ds_train = ds_pick.loc[idx_train]
  X_train = ds_train.loc[:, Xcols_ds_lin_pick]
  X_test = ds_test.loc[:, Xcols_ds_lin_pick]
#

  y_train = ds_train.loc[:, target]
  y_test = ds_test.loc[:, target]
  clf.fit(X_train, y_train)
  y_pred = clf.predict(X_test)

  ds_pick.loc[idx_test, target_pred] = y_pred

In [ ]:
mse = mean_squared_error(ds_pick[target].values, ds_pick[target_pred].values)
rmse = np.sqrt(mse)
rmse

In [ ]:
np.sqrt(mean_squared_error(ds_pick[target].values, ds_pick[target_pred].values, multioutput='raw_values'))

In [ ]:
ds_pick['err'] = [np.sqrt((x**2).sum()) for x in (ds_pick[Values_xyz].values-ds_pick[Values_xyz_pred].values)]
ds_pick['err'].describe()

In [ ]:
ds_pick.shape, ds_pick[ds_pick['err']>2].shape

# antRound

In [ ]:
ds_pick

In [ ]:
ds_pick.shape